<a href="https://colab.research.google.com/github/AlonAchituv/Programming-in-Generative-AI-Environments-Building-AI-Agent/blob/main/Shoes_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Block 1 — Installs (Colab)
!pip -q install -U langgraph typing_extensions
!pip -q install -U "langchain[openai]"
!pip -q install -U tavily-python
!pip -q install -U langchain-chroma chromadb
!pip -q install -U pypdf
!pip -q install -U langchain-community
!pip -q install -U psycopg2-binary
!pip -q install -U gradio
!pip -q install -U composio requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6

In [ ]:
# Block 2 — Imports + (optional) Google Drive mount
import os
import json
from uuid import uuid4
from datetime import datetime
from typing import Annotated, Optional, Literal, List, Dict, Any
from typing_extensions import TypedDict

import psycopg2
from contextlib import contextmanager

from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import MemorySaver

from tavily import TavilyClient

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from pathlib import Path
from composio import Composio

# Optional: Drive mount (Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)

Mounted at /content/drive
IN_COLAB: True


In [ ]:
# Block 3 — REPLACE your current "API Keys & Configuration" block with this full block
OPENAI_KEY_FILE = "/content/drive/MyDrive/openai_api_key.txt"
TAVILY_KEY_FILE = "/content/drive/MyDrive/tavily.txt"
LANGSMITH_KEY_FILE = "/content/drive/MyDrive/langsmith-api-key.txt"
SUPABASE_DB_PASSWORD_FILE = "/content/drive/MyDrive/supabase-Niv.txt"

# Optional — Email notification integration
SENDGRID_API_KEY_FILE = "/content/drive/MyDrive/sendgrid_api_key.txt"
COMPOSIO_API_KEY_FILE = "/content/drive/MyDrive/composio_api_key.txt"

COMPOSIO_USER_ID = "pg-test-8b781036-723c-4264-b5b2-356d08f83d7d"
SENDER_NAME = "ShoeOps Team"
SENDER_EMAIL = "pinguinefinalproject@gmail.com"
SENDGRID_MAIL_TOOL = "SENDGRID_SEND_EMAIL_WITH_TWILIO_SEND_GRID"
SENDGRID_TOOLKIT_VERSION = "20260312_00"

# --- DB CONFIG (only password is loaded from file) ---
DB_CONFIG = {
    "host": "aws-1-ap-south-1.pooler.supabase.com",
    "user": "postgres.qmhiygdchgihbqqjrgfn",
    "password": None,
    "port": "6543",
    "dbname": "postgres",
}

def _read_txt(path: str, encoding: str = "utf-8-sig") -> str:
    with open(path, "r", encoding=encoding) as f:
        return f.read().strip()

def _load_optional_env(var_name: str, file_path: str) -> bool:
    if not os.path.exists(file_path):
        return False
    value = _read_txt(file_path)
    if not value:
        return False
    os.environ[var_name] = value
    return True

def load_keys_and_config():
    # Required — OpenAI
    if not os.path.exists(OPENAI_KEY_FILE):
        raise FileNotFoundError(f"Missing OpenAI key file: {OPENAI_KEY_FILE}")
    os.environ["OPENAI_API_KEY"] = _read_txt(OPENAI_KEY_FILE)

    # Required — Supabase DB password
    if not os.path.exists(SUPABASE_DB_PASSWORD_FILE):
        raise FileNotFoundError(f"Missing Supabase DB password file: {SUPABASE_DB_PASSWORD_FILE}")
    DB_CONFIG["password"] = _read_txt(SUPABASE_DB_PASSWORD_FILE)

    # Optional — Tavily
    if os.path.exists(TAVILY_KEY_FILE):
        os.environ["TAVILY_API_KEY"] = _read_txt(TAVILY_KEY_FILE)

    # Optional — LangSmith
    if os.path.exists(LANGSMITH_KEY_FILE):
        os.environ["LANGCHAIN_TRACING_V2"] = "true"
        os.environ["LANGCHAIN_API_KEY"] = _read_txt(LANGSMITH_KEY_FILE)
        os.environ["LANGCHAIN_PROJECT"] = "ZAMSH_ShoeOps_Agent"
        print("✅ LangSmith tracing ON: ShoeOps_Agent")
    else:
        print("⚠️ LangSmith key file not found — tracing OFF")

    # Optional — SendGrid / Composio for deterministic notifications
    sendgrid_loaded = _load_optional_env("SENDGRID_API_KEY", SENDGRID_API_KEY_FILE)
    composio_loaded = _load_optional_env("COMPOSIO_API_KEY", COMPOSIO_API_KEY_FILE)

    print("✅ Keys loaded: OPENAI + DB password")
    print("✅ Tavily key present:", bool(os.environ.get("TAVILY_API_KEY")))
    print("✅ SendGrid key present:", sendgrid_loaded)
    print("✅ Composio key present:", composio_loaded)

load_keys_and_config()

✅ LangSmith tracing ON: ShoeOps_Agent
✅ Keys loaded: OPENAI + DB password
✅ Tavily key present: True
✅ SendGrid key present: True
✅ Composio key present: True


In [ ]:
# Block 4 — (LangSmith is now always-on from Cell 3 — this cell intentionally left minimal)
# Requirement: LangSmith tracing — set at startup, traces every graph.invoke() automatically
print("LangSmith tracing active:", os.environ.get("LANGCHAIN_TRACING_V2") == "true")
print("LangSmith project:", os.environ.get("LANGCHAIN_PROJECT", "NOT SET"))

LangSmith tracing active: True
LangSmith project: ZAMSH_ShoeOps_Agent


In [ ]:
# Debug LangSmith — run this AFTER Cell 3, BEFORE launching Gradio
import os

print("1. ENV VARS CHECK:")
print("   LANGCHAIN_TRACING_V2 =", repr(os.environ.get("LANGCHAIN_TRACING_V2")))
print("   LANGCHAIN_API_KEY    =", repr(os.environ.get("LANGCHAIN_API_KEY", "NOT SET")[:8]) + "...")
print("   LANGCHAIN_PROJECT    =", repr(os.environ.get("LANGCHAIN_PROJECT")))

print("\n2. KEY FILE CHECK:")
key_path = "/content/drive/MyDrive/langsmith-api-key-Niv.txt"
if os.path.exists(key_path):
    raw = open(key_path, "r", encoding="utf-8-sig").read()
    stripped = raw.strip()
    print(f"   File exists ✅")
    print(f"   Raw length: {len(raw)}, Stripped length: {len(stripped)}")
    print(f"   Starts with 'lsv2_': {stripped.startswith('lsv2_')}")
    print(f"   Has newlines: {chr(10) in raw}")
    print(f"   Has spaces: {' ' in stripped}")
    print(f"   First 12 chars: {repr(stripped[:12])}")
else:
    print(f"   ❌ File NOT FOUND at: {key_path}")

print("\n3. CONNECTIVITY TEST:")
try:
    from langsmith import Client
    client = Client()
    # This call will fail if the key is invalid
    projects = list(client.list_projects())
    print(f"   ✅ Connected! Found {len(projects)} existing projects:")
    for p in projects[:5]:
        print(f"      - {p.name}")
except Exception as e:
    print(f"   ❌ Connection failed: {type(e).__name__}: {e}")

1. ENV VARS CHECK:
   LANGCHAIN_TRACING_V2 = 'true'
   LANGCHAIN_API_KEY    = 'lsv2_pt_'...
   LANGCHAIN_PROJECT    = 'ZAMSH_ShoeOps_Agent'

2. KEY FILE CHECK:
   File exists ✅
   Raw length: 51, Stripped length: 51
   Starts with 'lsv2_': True
   Has newlines: False
   Has spaces: False
   First 12 chars: 'lsv2_pt_d6b7'

3. CONNECTIVITY TEST:
   ✅ Connected! Found 10 existing projects:
      - ZAMSH_ShoeOps_Agent
      - ShoeOps_Agent
      - Phone-Smart-8354
      - sample_exam_fixed
      - Ex_11_solution


In [ ]:
# Block 4 — RAG setup (policies PDFs -> Chroma)

POLICY_PDFS = [
    "/content/drive/MyDrive/ShoeOps_Transfer_SOP.pdf",
    "/content/drive/MyDrive/ShoeOps_Shipping_Policy.pdf",
    "/content/drive/MyDrive/ShoeOps_Return_Policy.pdf",
]

PERSIST_DIR = "/content/drive/MyDrive/chroma_shoeops"  # persistent across sessions (recommended)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vector_store = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embeddings,
)

In [ ]:
def _maybe_build_policy_index():
    # If collection already has docs, skip re-adding (simple dedupe avoidance)
    try:
        existing = vector_store._collection.count()
    except Exception:
        existing = 0

    if existing and existing > 0:
        print(f"✅ RAG index already has {existing} chunks. Skipping rebuild.")
        return

    # Load PDFs
    all_docs = []
    for f in POLICY_PDFS:
        if not os.path.exists(f):
            print(f"⚠️ Missing PDF: {f}")
            continue
        loader = PyPDFLoader(f)
        docs = loader.load()
        for d in docs:
            d.metadata["source_file"] = os.path.basename(f)
        all_docs.extend(docs)

    if not all_docs:
        print("⚠️ No policy PDFs loaded. RAG tool will return 'no docs'.")
        return

    splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=150)
    splits = splitter.split_documents(all_docs)

    # Add once
    ids = [str(uuid4()) for _ in splits]
    vector_store.add_documents(splits, ids=ids)

    print(f"✅ Built RAG index: {len(all_docs)} pages -> {len(splits)} chunks")

_maybe_build_policy_index()

✅ RAG index already has 39 chunks. Skipping rebuild.


In [ ]:
# Block 5 — DB helpers + Tools (READ + WRITE)

def get_db_conn():
    """Create a new DB connection (caller must close it)."""
    return psycopg2.connect(**DB_CONFIG)

In [ ]:
# ============================================================
# Tool: query_stores + startup business context globals
# Requirement: Structured Data Access (READ from stores table)
# ============================================================

@tool
def query_stores(city: str = None) -> dict:
    """List all stores, optionally filtered by city name (partial match).
    Returns store_id, name, and city for each store.
    Use this FIRST whenever a user refers to a store by city name or nickname."""
    conn = get_db_conn()
    try:
        cur = conn.cursor()
        if city:
            cur.execute(
                """
                SELECT store_id, name, city
                FROM stores
                WHERE LOWER(city) LIKE LOWER(%s) OR LOWER(name) LIKE LOWER(%s)
                ORDER BY store_id
                """,
                (f"%{city.strip()}%", f"%{city.strip()}%"),
            )
        else:
            cur.execute("SELECT store_id, name, city FROM stores ORDER BY store_id")
        rows = cur.fetchall()
    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    return {
        "success": True,
        "count": len(rows),
        "stores": [
            {"store_id": r[0], "name": r[1], "city": r[2]}
            for r in rows
        ],
    }

In [ ]:
# Block — Tool: check if username is already taken (used during guest registration)
# Requirement: Structured Data Access (READ from users table)

@tool
def check_username_exists(username: str) -> dict:
    """Check whether a username already exists in the users table.
    Returns {"exists": True/False, "username": ...}.
    Used during guest self-registration to prevent duplicates."""
    if not username or not username.strip():
        return {"exists": False, "error": "Empty username provided."}

    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute("SELECT 1 FROM users WHERE LOWER(username) = LOWER(%s)", (username.strip(),))
        found = cur.fetchone() is not None
    except Exception as e:
        return {"exists": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    return {"exists": found, "username": username.strip()}

In [ ]:
# Block — Tool: register a new customer account (WRITE — self-service, no HITL)
# Requirement: Write Operation — inserts row into Supabase users table

@tool
def register_new_user(username: str, pin_code: str, display_name: str, email: str) -> dict:
    """WRITE: Create a new customer account in the users table.
    Auto-generates user_id (U-XXX) and customer_id (CUST-XXX).
    Returns the new user record on success."""
    username = (username or "").strip()
    pin_code = (pin_code or "").strip()
    display_name = (display_name or "").strip()
    email = (email or "").strip()

    if not username or not pin_code or not display_name or not email:
        return {"success": False, "error": "All fields required: username, pin_code, display_name, email."}

    conn = get_db_conn()
    try:
        cur = conn.cursor()

        # 1) Check username not taken
        cur.execute("SELECT 1 FROM users WHERE LOWER(username) = LOWER(%s)", (username,))
        if cur.fetchone():
            return {"success": False, "error": f"Username '{username}' is already taken."}

        # 2) Check email not taken
        cur.execute("SELECT 1 FROM users WHERE LOWER(email) = LOWER(%s)", (email,))
        if cur.fetchone():
            return {"success": False, "error": f"Email '{email}' is already in use."}

        # 3) Generate next user_id (U-XXX)
        cur.execute("SELECT user_id FROM users WHERE user_id LIKE 'U-%' ORDER BY user_id DESC LIMIT 1")
        row = cur.fetchone()
        if row:
            last_num = int(row[0].split("-")[1])
            new_user_id = f"U-{last_num + 1:03d}"
        else:
            new_user_id = "U-001"

        # 4) Generate next customer_id (CUST-XXX)
        cur.execute("SELECT customer_id FROM users WHERE customer_id LIKE 'CUST-%' ORDER BY customer_id DESC LIMIT 1")
        row = cur.fetchone()
        if row:
            last_num = int(row[0].split("-")[1])
            new_customer_id = f"CUST-{last_num + 1:03d}"
        else:
            new_customer_id = "CUST-001"

        # 5) INSERT
        cur.execute(
            """INSERT INTO users (user_id, username, email, pin_code, role, display_name, customer_id)
               VALUES (%s, %s, %s, %s, 'customer', %s, %s)""",
            (new_user_id, username, email, pin_code, display_name, new_customer_id),
        )
        conn.commit()

        return {
            "success": True,
            "user_id": new_user_id,
            "username": username,
            "email": email,
            "customer_id": new_customer_id,
            "display_name": display_name,
            "role": "customer",
        }

    except Exception as e:
        conn.rollback()
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

In [ ]:
@tool
def authenticate_user(username: str, pin_code: str) -> dict:
    """Authenticate user from users table by username + pin_code."""
    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute(
            """
            SELECT user_id, username, email, role, display_name, customer_id
            FROM users
            WHERE username=%s AND pin_code=%s
            """,
            (username, pin_code),
        )
        row = cur.fetchone()
    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    if not row:
        return {"success": False, "error": "Invalid username or PIN"}

    user_id, username, email, role, display_name, customer_id = row
    return {
        "success": True,
        "user": {
            "user_id": user_id,
            "username": username,
            "email": email,
            "role": role,
            "display_name": display_name,
            "customer_id": customer_id,
        },
    }

In [ ]:
# ============================================================
# NEW CELL — Add right after Cell 10 (after authenticate_user)
# Tool: lookup_customer — lets managers find a customer by name/email/ID
# Requirement: Structured Data Access (READ from users table)
# ============================================================

@tool
def lookup_customer(search_term: str) -> dict:
    """Manager-only: Look up a customer by name, email, or customer_id.
    Returns matching customer records (display_name, customer_id, email).
    Does NOT expose sensitive fields like pin_code."""
    if not search_term or not search_term.strip():
        return {"success": False, "error": "Please provide a customer name, email, or customer ID to search."}

    term = search_term.strip()
    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute(
            """
            SELECT customer_id, display_name, email, username
            FROM users
            WHERE role = 'customer'
              AND (
                LOWER(display_name) LIKE LOWER(%s)
                OR LOWER(email) LIKE LOWER(%s)
                OR LOWER(customer_id) = LOWER(%s)
                OR LOWER(username) LIKE LOWER(%s)
              )
            ORDER BY display_name
            LIMIT 10
            """,
            (f"%{term}%", f"%{term}%", term, f"%{term}%"),
        )
        rows = cur.fetchall()
    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    if not rows:
        return {"success": False, "error": f"No customer found matching '{term}'."}

    customers = [
        {
            "customer_id": r[0],
            "display_name": r[1],
            "email": r[2],
            "username": r[3],
        }
        for r in rows
    ]

    # MULTIPLE MATCHES: return explicit instruction to ask for clarification
    if len(customers) > 1:
        return {
            "success": True,
            "count": len(customers),
            "customers": customers,
            "IMPORTANT": (
                f"MULTIPLE MATCHES FOUND ({len(customers)}). "
                "You MUST list ALL matches below and ask the manager to specify which customer. "
                "Do NOT pick one automatically. Present each match with their customer_id and display_name."
            ),
        }

    return {
        "success": True,
        "count": 1,
        "customers": customers,
    }

In [ ]:
# Block R4 — query_store_availability (supports both directions)
# Customer-safe: by SKU (which stores have it?) OR by store (what's available there?)

@tool
def query_store_availability(sku: str = None, store_id: str = None) -> dict:
    """Customer-safe availability check.
    - sku only: list stores where this SKU has available stock (>0).
    - store_id only: list all products with available stock at that store.
    - both: check specific product at specific store.
    At least one of sku or store_id is required."""
    if not sku and not store_id:
        return {"success": False, "error": "Provide at least one of: sku, store_id"}

    conn = get_db_conn()
    try:
        cur = conn.cursor()

        wh = []
        params = []
        if sku:
            wh.append("i.sku = %s"); params.append(sku)
        if store_id:
            wh.append("i.store_id = %s"); params.append(store_id)

        where_sql = " AND ".join(wh)

        cur.execute(
            f"""
            SELECT i.store_id, i.sku, p.model, p.color, p.size, p.price,
                   COALESCE(i.on_hand,0) AS on_hand, COALESCE(i.reserved,0) AS reserved
            FROM inventory i
            JOIN products p ON i.sku = p.sku
            WHERE {where_sql}
            ORDER BY i.store_id, p.model, p.size
            """,
            tuple(params),
        )
        rows = cur.fetchall()
    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    results = []
    for store, sku_val, model, color, size, price, on_hand, reserved in rows:
        available = int(on_hand) - int(reserved)
        if available > 0:
            results.append({
                "store_id": store,
                "sku": sku_val,
                "model": model,
                "color": color,
                "size": size,
                "price": float(price),
                "available": available,
            })

    return {
        "success": True,
        "sku": sku,
        "store_id": store_id,
        "count": len(results),
        "items": results,
    }

In [ ]:
# Block 1 — query_order_customer (fixed)
@tool
def query_order_customer(order_id: str, customer_id: str) -> dict:
    """Customer-safe: only returns order if it belongs to customer_id."""
    if not order_id or not customer_id:
        return {"success": False, "error": "Missing order_id or customer_id"}

    conn = get_db_conn()
    try:
        cur = conn.cursor()

        cur.execute(
            """
            SELECT order_id, store_id, channel, status, customer_name, customer_id, created_at
            FROM orders
            WHERE order_id=%s AND customer_id=%s
            """,
            (order_id, customer_id),
        )
        order_row = cur.fetchone()
        if not order_row:
            return {"success": False, "error": "Order not found or access denied"}

        cur.execute(
            """
            SELECT oi.sku, oi.qty, p.model, p.color, p.size, p.price
            FROM order_items oi
            JOIN products p ON p.sku = oi.sku
            WHERE oi.order_id=%s
            """
            ,
            (order_id,),
        )
        items = cur.fetchall()

    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    order = {
        "order_id": order_row[0],
        "store_id": order_row[1],
        "channel": order_row[2],
        "status": order_row[3],
        "customer_name": order_row[4],
        "customer_id": order_row[5],
        "created_at": str(order_row[6]),
    }

    order_items = [
        {
            "sku": r[0],
            "qty": r[1],
            "product": {
                "model": r[2],
                "color": r[3],
                "size": r[4],
                "price": float(r[5]),
            },
        }
        for r in items
    ]

    return {"success": True, "order": order, "items": order_items}

In [ ]:
# Block 2 — query_order_manager (your version is correct)
@tool
def query_order_manager(order_id: str) -> dict:
    """Manager: can view any order."""
    if not order_id:
        return {"success": False, "error": "Missing order_id"}

    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute(
            """
            SELECT order_id, store_id, channel, status, customer_name, customer_id, created_at
            FROM orders
            WHERE order_id=%s
            """,
            (order_id,),
        )
        order_row = cur.fetchone()
        if not order_row:
            return {"success": False, "error": "Order not found"}

        cur.execute(
            """
            SELECT oi.sku, oi.qty, p.model, p.color, p.size, p.price
            FROM order_items oi
            JOIN products p ON p.sku = oi.sku
            WHERE oi.order_id=%s
            """,
            (order_id,),
        )
        items = cur.fetchall()

    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    order = {
        "order_id": order_row[0],
        "store_id": order_row[1],
        "channel": order_row[2],
        "status": order_row[3],
        "customer_name": order_row[4],
        "customer_id": order_row[5],
        "created_at": str(order_row[6]),
    }

    order_items = [
        {
            "sku": r[0],
            "qty": r[1],
            "product": {
                "model": r[2],
                "color": r[3],
                "size": r[4],
                "price": float(r[5]),
            },
        }
        for r in items
    ]

    return {"success": True, "order": order, "items": order_items}

In [ ]:
@tool
def query_products(sku: str = None, model: str = None, color: str = None, size: int = None) -> dict:
    """Read products from DB. Provide sku or filters."""
    wh = []
    params = []

    if sku:
        wh.append("sku=%s"); params.append(sku)
    if model:
        wh.append("LOWER(model)=LOWER(%s)"); params.append(model)
    if color:
        wh.append("LOWER(color)=LOWER(%s)"); params.append(color)
    if size is not None:
        wh.append("size=%s"); params.append(size)

    where_sql = ("WHERE " + " AND ".join(wh)) if wh else ""
    sql = f"SELECT sku, model, color, size, price FROM products {where_sql} ORDER BY model, color, size LIMIT 50"

    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute(sql, tuple(params))
        rows = cur.fetchall()
    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    return {
        "success": True,
        "count": len(rows),
        "products": [
            {"sku": r[0], "model": r[1], "color": r[2], "size": r[3], "price": float(r[4])}
            for r in rows
        ],
    }

In [ ]:
# Block 3 — list_my_orders (your version is good)
@tool
def list_my_orders(customer_id: str, limit: int = 10) -> dict:
    """Read recent orders for the logged-in customer (ownership enforced)."""
    if not customer_id:
        return {"success": False, "error": "Missing customer_id (not logged in as a customer)."}

    limit = int(limit or 10)
    limit = max(1, min(limit, 50))

    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute(
            """
            SELECT order_id, store_id, channel, status, created_at
            FROM orders
            WHERE customer_id=%s
            ORDER BY created_at DESC
            LIMIT %s
            """,
            (customer_id, limit),
        )
        rows = cur.fetchall()
    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    return {
        "success": True,
        "count": len(rows),
        "orders": [
            {
                "order_id": r[0],
                "store_id": r[1],
                "channel": r[2],
                "status": r[3],
                "created_at": str(r[4]),
            }
            for r in rows
        ],
    }

In [ ]:
@tool
def query_inventory(sku: str = None, store_id: str = None) -> dict:
    """Read inventory. available=on_hand-reserved."""
    wh = []
    params = []
    if sku:
        wh.append("i.sku=%s"); params.append(sku)
    if store_id:
        wh.append("i.store_id=%s"); params.append(store_id)

    where_sql = ("WHERE " + " AND ".join(wh)) if wh else ""
    sql = f"""
        SELECT i.store_id, s.name, s.city, i.sku, i.on_hand, i.reserved, (i.on_hand - i.reserved) AS available
        FROM inventory i
        JOIN stores s ON s.store_id = i.store_id
        {where_sql}
        ORDER BY i.store_id, i.sku
        LIMIT 200
    """

    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute(sql, tuple(params))
        rows = cur.fetchall()
    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    return {
        "success": True,
        "count": len(rows),
        "rows": [
            {
                "store_id": r[0],
                "store_name": r[1],
                "city": r[2],
                "sku": r[3],
                "on_hand": r[4],
                "reserved": r[5],
                "available": r[6],
            }
            for r in rows
        ],
    }

In [ ]:
@tool
def search_policy(query: str) -> str:
    """RAG search on policy PDFs."""
    results = vector_store.similarity_search(query, k=3)
    if not results:
        return "No relevant policy found (no docs indexed / no match)."

    out = []
    for i, doc in enumerate(results, 1):
        src = doc.metadata.get("source_file") or doc.metadata.get("source") or "unknown"
        page = doc.metadata.get("page", "N/A")
        snippet = doc.page_content[:500].replace("\n", " ")
        out.append(f"[{i}] {src} (page {page}) — {snippet}")
    return "\n\n".join(out)

In [ ]:
@tool
def tavily_search(query: str) -> str:
    """WEB search (Tavily). Returns short bullet summaries."""
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "WEB search is not configured (missing TAVILY_API_KEY)."
    client = TavilyClient(api_key=api_key)
    try:
        res = client.search(query=query, max_results=5)
        items = res.get("results", []) if isinstance(res, dict) else []
        lines = []
        for it in items[:5]:
            title = it.get("title", "")
            content = (it.get("content") or "")[:220].replace("\n", " ")
            lines.append(f"- {title}: {content}")
        return "\n".join(lines) if lines else "No WEB results."
    except Exception as e:
        return f"WEB search failed: {e}"

In [ ]:
# Block — Order tools: prepare_order_summary + execute_customer_order
# Requirement: Write Operation + Structured Data Access

@tool
def prepare_order_summary(items_json: str, store_id: str) -> dict:
    """Check stock availability for a list of items at a specific store.
    items_json: JSON string like [{"sku":"ADI-UB5-W-42","qty":1}, ...]
    Returns summary with availability, prices, and total."""
    try:
        items = json.loads(items_json)
    except Exception:
        return {"success": False, "error": "Invalid items_json format. Must be JSON list of {sku, qty}."}

    if not items:
        return {"success": False, "error": "Items list is empty."}
    if not store_id:
        return {"success": False, "error": "store_id is required."}

    conn = get_db_conn()
    try:
        cur = conn.cursor()

        # Verify store exists
        cur.execute("SELECT name, city FROM stores WHERE store_id = %s", (store_id,))
        store_row = cur.fetchone()
        if not store_row:
            return {"success": False, "error": f"Store '{store_id}' not found."}
        store_name, store_city = store_row

        result_items = []
        unavailable = []
        total_price = 0.0

        for item in items:
            sku = (item.get("sku") or "").strip()
            qty = int(item.get("qty", 1))

            if not sku or qty < 1:
                unavailable.append({"sku": sku, "qty": qty, "reason": "Invalid SKU or qty."})
                continue

            # Get product details
            cur.execute("SELECT model, color, size, price FROM products WHERE sku = %s", (sku,))
            prod = cur.fetchone()
            if not prod:
                unavailable.append({"sku": sku, "qty": qty, "reason": f"Product {sku} not found."})
                continue

            model, color, size, price = prod

            # Check inventory at this store
            cur.execute(
                "SELECT on_hand, reserved FROM inventory WHERE store_id = %s AND sku = %s",
                (store_id, sku),
            )
            inv = cur.fetchone()
            if inv:
                on_hand, reserved = inv[0] or 0, inv[1] or 0
                available = on_hand - reserved
            else:
                available = 0

            in_stock = available >= qty
            line_total = float(price) * qty

            item_result = {
                "sku": sku,
                "model": model,
                "color": color,
                "size": size,
                "qty": qty,
                "unit_price": float(price),
                "line_total": line_total,
                "available": available,
                "in_stock": in_stock,
            }

            result_items.append(item_result)

            if in_stock:
                total_price += line_total
            else:
                unavailable.append({
                    "sku": sku, "qty": qty, "available": available,
                    "reason": f"Only {available} available, requested {qty}.",
                })

        all_available = len(unavailable) == 0 and len(result_items) > 0

        return {
            "success": True,
            "all_available": all_available,
            "store_id": store_id,
            "store_name": f"{store_name} ({store_city})",
            "items": result_items,
            "total_price": round(total_price, 2),
            "unavailable_items": unavailable,
        }

    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

In [ ]:
# ============================================================
# REPLACE Cell 20 — execute_customer_order (with customer_id validation)
# Requirement: Write Operation — inserts row into Supabase orders table
# FIX: Validates that customer_id exists in users table BEFORE inserting
# ============================================================

@tool
def execute_customer_order(items_json: str, store_id: str, customer_id: str, customer_name: str) -> dict:
    """WRITE: Place a customer order. Inserts into orders + order_items, updates inventory reserved.
    Sends a deterministic confirmation email after DB success. If email fails, the order still succeeds.
    IMPORTANT: customer_id must be a valid CUST-XXX ID from the users table."""
    try:
        items = json.loads(items_json)
    except Exception:
        return {"success": False, "error": "Invalid items_json format."}

    if not items:
        return {"success": False, "error": "Items list is empty."}
    if not store_id or not customer_id or not customer_name:
        return {"success": False, "error": "store_id, customer_id, and customer_name are all required."}

    conn = get_db_conn()
    try:
        cur = conn.cursor()

        # ── NEW: Validate customer_id exists in the users table ──
        cur.execute(
            "SELECT customer_id, display_name FROM users WHERE customer_id = %s",
            (customer_id,),
        )
        cust_row = cur.fetchone()
        if not cust_row:
            return {
                "success": False,
                "error": (
                    f"Customer ID '{customer_id}' does not exist in the system. "
                    f"Use the lookup_customer tool to find the correct customer_id (format: CUST-XXX) first."
                ),
            }
        # Use the verified display_name from DB for consistency
        verified_customer_name = cust_row[1] or customer_name

        # 1) Generate order_id
        cur.execute("SELECT order_id FROM orders WHERE order_id LIKE 'ORD-%' ORDER BY order_id DESC LIMIT 1")
        row = cur.fetchone()
        if row:
            last_num = int(row[0].split("-")[1])
            order_id = f"ORD-{last_num + 1}"
        else:
            order_id = "ORD-10001"

        # 2) Verify stock for each item
        total_price = 0.0
        verified_items = []

        for item in items:
            sku = (item.get("sku") or "").strip()
            qty = int(item.get("qty", 1))

            cur.execute(
                "SELECT on_hand, reserved FROM inventory WHERE store_id = %s AND sku = %s",
                (store_id, sku),
            )
            inv = cur.fetchone()
            if not inv:
                conn.rollback()
                return {"success": False, "error": f"No inventory record for {sku} at {store_id}."}

            on_hand, reserved = inv[0] or 0, inv[1] or 0
            available = on_hand - reserved

            if available < qty:
                conn.rollback()
                return {
                    "success": False,
                    "error": f"Insufficient stock for {sku} at {store_id}. Available: {available}, requested: {qty}.",
                }

            cur.execute("SELECT price FROM products WHERE sku = %s", (sku,))
            price_row = cur.fetchone()
            unit_price = float(price_row[0]) if price_row else 0.0
            total_price += unit_price * qty

            verified_items.append({"sku": sku, "qty": qty, "unit_price": unit_price})

        # 3) INSERT order (use verified_customer_name from DB)
        cur.execute(
            """INSERT INTO orders (order_id, store_id, channel, status, customer_name, customer_id)
               VALUES (%s, %s, 'chatbot', 'PENDING', %s, %s)""",
            (order_id, store_id, verified_customer_name, customer_id),
        )

        # 4) INSERT order_items + UPDATE inventory reserved
        for vi in verified_items:
            cur.execute(
                "INSERT INTO order_items (order_id, sku, qty) VALUES (%s, %s, %s)",
                (order_id, vi["sku"], vi["qty"]),
            )
            cur.execute(
                "UPDATE inventory SET reserved = reserved + %s WHERE store_id = %s AND sku = %s",
                (vi["qty"], store_id, vi["sku"]),
            )

        conn.commit()

    except Exception as e:
        conn.rollback()
        return {"success": False, "error": f"DB error: {type(e).__name__}: {e}"}
    finally:
        conn.close()

    email_result = send_order_created_notification(
        order_id=order_id,
        customer_id=customer_id,
        customer_name=verified_customer_name,
        store_id=store_id,
        items=verified_items,
        total_price=round(total_price, 2),
    )

    warning_text = None
    if not email_result.get("success"):
        warning_text = f"The confirmation email was not sent: {email_result.get('error', 'unknown email error')}"

    return {
        "success": True,
        "order_id": order_id,
        "store_id": store_id,
        "customer_id": customer_id,
        "customer_name": verified_customer_name,
        "items": verified_items,
        "total_price": round(total_price, 2),
        "status": "PENDING",
        "email_sent": bool(email_result.get("success")),
        "email_warning": warning_text,
    }

In [ ]:
@tool
def log_audit(action_type: str, payload_json: dict, approved_by: str) -> dict:
    """Write audit log row."""
    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute(
            """
            INSERT INTO audit_log(action_type, payload_json, approved_by, created_at)
            VALUES (%s, %s::jsonb, %s, NOW())
            """,
            (action_type, json.dumps(payload_json), approved_by),
        )
        conn.commit()
        return {"success": True}
    except Exception as e:
        conn.rollback()
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

In [ ]:
# Block 1 — FIX execute_transfer: atomic dest UPSERT + DB readback (prevents wrong available)
@tool
def execute_transfer(from_store: str, to_store: str, sku: str, qty: int, approved_by: str) -> dict:
    """WRITE: move stock immediately between stores (on_hand), and log audit."""
    if not from_store or not to_store or not sku:
        return {"success": False, "error": "Missing from_store/to_store/sku"}
    if from_store == to_store:
        return {"success": False, "error": "from_store and to_store must be different"}
    if qty is None:
        return {"success": False, "error": "qty is required"}
    try:
        qty = int(qty)
    except Exception:
        return {"success": False, "error": "qty must be an integer"}
    if qty < 1:
        return {"success": False, "error": "qty must be >= 1"}
    if qty > 20:
        return {"success": False, "error": "qty exceeds max transfer limit (20)"}

    conn = get_db_conn()
    try:
        cur = conn.cursor()

        # 1) Lock + verify source
        cur.execute(
            "SELECT on_hand, reserved FROM inventory WHERE store_id=%s AND sku=%s FOR UPDATE",
            (from_store, sku),
        )
        row = cur.fetchone()
        if not row:
            conn.rollback()
            return {"success": False, "error": f"No inventory record for {sku} in {from_store}"}

        on_hand_src, reserved_src = row[0] or 0, row[1] or 0
        available_before = int(on_hand_src) - int(reserved_src)
        if available_before < qty:
            conn.rollback()
            return {"success": False, "error": f"Insufficient available stock. Available={available_before}, Requested={qty}"}

        # 2) Apply source decrement
        cur.execute(
            """
            UPDATE inventory
            SET on_hand = COALESCE(on_hand,0) - %s
            WHERE store_id=%s AND sku=%s
            RETURNING COALESCE(on_hand,0), COALESCE(reserved,0)
            """,
            (qty, from_store, sku),
        )
        src_after = cur.fetchone()
        if not src_after:
            conn.rollback()
            return {"success": False, "error": "Failed to update source inventory row."}

        # 3) Destination UPSERT increment (requires UNIQUE(store_id, sku))
        cur.execute(
            """
            INSERT INTO inventory(store_id, sku, on_hand, reserved)
            VALUES (%s, %s, %s, 0)
            ON CONFLICT (store_id, sku)
            DO UPDATE SET on_hand = COALESCE(inventory.on_hand,0) + EXCLUDED.on_hand
            RETURNING COALESCE(on_hand,0), COALESCE(reserved,0)
            """,
            (to_store, sku, qty),
        )
        dst_after = cur.fetchone()
        if not dst_after:
            conn.rollback()
            return {"success": False, "error": "Failed to upsert destination inventory row."}

        conn.commit()

    except Exception as e:
        conn.rollback()
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    src_on_hand_after, src_reserved_after = int(src_after[0]), int(src_after[1])
    dst_on_hand_after, dst_reserved_after = int(dst_after[0]), int(dst_after[1])

    from_available_after = src_on_hand_after - src_reserved_after
    to_available_after = dst_on_hand_after - dst_reserved_after

    _ = log_audit.invoke({
        "action_type": "TRANSFER",
        "payload_json": {
            "from_store": from_store,
            "to_store": to_store,
            "sku": sku,
            "qty": qty,
            "from_available_before": available_before,
            "from_available_after": from_available_after,
            "to_on_hand_after": dst_on_hand_after,
            "to_reserved_after": dst_reserved_after,
            "to_available_after": to_available_after,
        },
        "approved_by": approved_by,
    })

    return {
        "success": True,
        "message": f"Moved {qty} units of {sku} from {from_store} to {to_store}",
        "from_available_before": available_before,
        "from_available_after": from_available_after,
        "to_on_hand_after": dst_on_hand_after,
        "to_reserved_after": dst_reserved_after,
        "to_available_after": to_available_after,
    }

In [ ]:
# Block 6 — REPLACE update_order_status with this full function
@tool
def update_order_status(order_id: str, new_status: str, approved_by: str) -> dict:
    """WRITE: update order status and log audit.
    Sends a deterministic notification email after DB success. If email fails, the status update still succeeds."""
    if not order_id or not new_status or not approved_by:
        return {"success": False, "error": "Missing order_id, new_status, or approved_by"}

    conn = get_db_conn()
    try:
        cur = conn.cursor()

        cur.execute(
            """
            SELECT status, store_id, customer_id, customer_name
            FROM orders
            WHERE order_id=%s
            """,
            (order_id,),
        )
        row = cur.fetchone()
        if not row:
            return {"success": False, "error": f"Order {order_id} not found"}

        old_status = row[0]
        store_id = row[1]
        customer_id = row[2]
        customer_name = row[3]

        cur.execute(
            """
            UPDATE orders
            SET status=%s
            WHERE order_id=%s
            """,
            (new_status, order_id),
        )

        conn.commit()

    except Exception as e:
        conn.rollback()
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

    _ = log_audit.invoke({
        "action_type": "STATUS_UPDATE",
        "payload_json": {
            "order_id": order_id,
            "store_id": store_id,
            "old_status": old_status,
            "new_status": new_status,
        },
        "approved_by": approved_by,
    })

    email_result = send_status_update_notification(
        order_id=order_id,
        customer_id=customer_id,
        customer_name=customer_name,
        old_status=old_status,
        new_status=new_status,
        store_id=store_id,
    )

    warning_text = None
    if not email_result.get("success"):
        warning_text = f"The status update email was not sent: {email_result.get('error', 'unknown email error')}"

    return {
        "success": True,
        "message": f"Order {order_id} status updated",
        "order_id": order_id,
        "store_id": store_id,
        "customer_id": customer_id,
        "customer_name": customer_name,
        "old_status": old_status,
        "new_status": new_status,
        "email_sent": bool(email_result.get("success")),
        "warning_text": warning_text,
    }

In [ ]:
# Block 4 — ADD this new helper block AFTER update_order_status and BEFORE llm = ChatOpenAI(...)

EMAIL_NOTIFICATIONS_ENABLED = bool(os.environ.get("COMPOSIO_API_KEY"))
composio_client = None

if EMAIL_NOTIFICATIONS_ENABLED:
    try:
        composio_client = Composio(
            api_key=os.environ["COMPOSIO_API_KEY"],
            toolkit_versions={"sendgrid": SENDGRID_TOOLKIT_VERSION},
        )
        print("✅ Composio email client initialized.")
    except Exception as e:
        composio_client = None
        EMAIL_NOTIFICATIONS_ENABLED = False
        print(f"⚠️ Composio email client init failed: {e}")
else:
    print("⚠️ Email notifications OFF (missing COMPOSIO_API_KEY).")


✅ Composio email client initialized.


In [ ]:
def send_email_via_composio(
    to_email: str,
    subject: str,
    html_body: str,
    from_email: str = SENDER_EMAIL,
    from_name: str = SENDER_NAME,
    to_name: Optional[str] = None,
) -> dict:
    if not to_email:
        return {"success": False, "error": "Missing recipient email."}

    if not EMAIL_NOTIFICATIONS_ENABLED or composio_client is None:
        return {"success": False, "error": "Email notifications are not configured."}

    arguments = {
        "from__email": from_email,
        "from__name": from_name,
        "subject": subject,
        "content": [
            {
                "type": "text/html",
                "value": html_body,
            }
        ],
        "personalizations": [
            {
                "to": [
                    {"email": to_email, "name": to_name} if to_name else {"email": to_email}
                ]
            }
        ],
    }

    try:
        result = composio_client.tools.execute(
            SENDGRID_MAIL_TOOL,
            user_id=COMPOSIO_USER_ID,
            arguments=arguments,
            version=SENDGRID_TOOLKIT_VERSION,
        )
        return {
            "success": True,
            "tool": SENDGRID_MAIL_TOOL,
            "version": SENDGRID_TOOLKIT_VERSION,
            "raw_result": result,
        }
    except Exception as e:
        return {
            "success": False,
            "tool": SENDGRID_MAIL_TOOL,
            "version": SENDGRID_TOOLKIT_VERSION,
            "error": str(e),
        }

In [ ]:
def get_customer_contact_by_customer_id(customer_id: str) -> dict:
    if not customer_id:
        return {"success": False, "error": "Missing customer_id."}

    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute(
            """
            SELECT customer_id, display_name, email
            FROM users
            WHERE customer_id=%s
            LIMIT 1
            """,
            (customer_id,),
        )
        row = cur.fetchone()
        if not row:
            return {"success": False, "error": f"No user found for customer_id {customer_id}."}

        return {
            "success": True,
            "customer_id": row[0],
            "display_name": row[1],
            "email": row[2],
        }
    except Exception as e:
        return {"success": False, "error": f"DB error: {e}"}
    finally:
        conn.close()

In [ ]:
def _format_currency(amount: float) -> str:
    return f"${float(amount):,.2f}"

In [ ]:
# ============================================================
# REPLACE Cell 28 — build_order_created_email_html (Professional Styled)
# ============================================================

def build_order_created_email_html(order_id: str, customer_name: str, store_id: str, items: List[dict], total_price: float) -> str:
    # Build item rows for the order table
    item_rows = ""
    for item in items:
        qty = int(item.get('qty', 0))
        sku = item.get('sku', '—')
        unit_price = float(item.get('unit_price', 0.0))
        line_total = qty * unit_price
        item_rows += f"""
            <tr>
                <td style="padding: 12px 16px; border-bottom: 1px solid #eef0f3; font-family: 'Segoe UI', Arial, sans-serif; font-size: 14px; color: #374151;">{sku}</td>
                <td style="padding: 12px 16px; border-bottom: 1px solid #eef0f3; font-family: 'Segoe UI', Arial, sans-serif; font-size: 14px; color: #374151; text-align: center;">{qty}</td>
                <td style="padding: 12px 16px; border-bottom: 1px solid #eef0f3; font-family: 'Segoe UI', Arial, sans-serif; font-size: 14px; color: #374151; text-align: right;">{_format_currency(unit_price)}</td>
                <td style="padding: 12px 16px; border-bottom: 1px solid #eef0f3; font-family: 'Segoe UI', Arial, sans-serif; font-size: 14px; color: #374151; text-align: right; font-weight: 600;">{_format_currency(line_total)}</td>
            </tr>"""

    if not item_rows:
        item_rows = """
            <tr>
                <td colspan="4" style="padding: 16px; text-align: center; color: #6b7280; font-style: italic;">Your order items were recorded successfully.</td>
            </tr>"""

    return f"""
    <div style="max-width: 600px; margin: 0 auto; background-color: #ffffff; font-family: 'Segoe UI', Arial, sans-serif;">

        <!-- Header Banner -->
        <div style="background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%); padding: 32px 40px; text-align: center; border-radius: 8px 8px 0 0;">
            <div style="font-size: 32px; margin-bottom: 8px;">👟</div>
            <h1 style="margin: 0; font-size: 24px; font-weight: 700; color: #ffffff; letter-spacing: 0.5px;">ShoeOps</h1>
            <p style="margin: 4px 0 0; font-size: 13px; color: #94a3b8; letter-spacing: 1px; text-transform: uppercase;">Order Confirmation</p>
        </div>

        <!-- Success Badge -->
        <div style="background-color: #f0fdf4; border-left: 4px solid #22c55e; padding: 16px 24px; margin: 24px 32px 0;">
            <p style="margin: 0; font-size: 15px; color: #166534; font-weight: 600;">
                ✅ &nbsp;Order placed successfully!
            </p>
        </div>

        <!-- Body -->
        <div style="padding: 24px 32px;">

            <p style="font-size: 15px; color: #374151; line-height: 1.6; margin: 0 0 20px;">
                Hi <strong>{customer_name}</strong>,
            </p>
            <p style="font-size: 15px; color: #374151; line-height: 1.6; margin: 0 0 24px;">
                Thank you for your order! Here are your details:
            </p>

            <!-- Order Info Card -->
            <div style="background-color: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px 24px; margin-bottom: 24px;">
                <table style="width: 100%; border-collapse: collapse;">
                    <tr>
                        <td style="padding: 6px 0; font-size: 13px; color: #6b7280; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600;">Order ID</td>
                        <td style="padding: 6px 0; font-size: 15px; color: #1e293b; font-weight: 700; text-align: right;">{order_id}</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-size: 13px; color: #6b7280; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600;">Pickup Store</td>
                        <td style="padding: 6px 0; font-size: 15px; color: #1e293b; font-weight: 600; text-align: right;">{store_id}</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-size: 13px; color: #6b7280; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600;">Status</td>
                        <td style="padding: 6px 0; text-align: right;">
                            <span style="display: inline-block; background-color: #fef3c7; color: #92400e; font-size: 12px; font-weight: 700; padding: 3px 12px; border-radius: 12px; text-transform: uppercase; letter-spacing: 0.5px;">Pending</span>
                        </td>
                    </tr>
                </table>
            </div>

            <!-- Items Table -->
            <table style="width: 100%; border-collapse: collapse; margin-bottom: 16px;">
                <thead>
                    <tr style="background-color: #f1f5f9;">
                        <th style="padding: 10px 16px; font-size: 12px; color: #64748b; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 700; text-align: left; border-bottom: 2px solid #e2e8f0;">Item</th>
                        <th style="padding: 10px 16px; font-size: 12px; color: #64748b; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 700; text-align: center; border-bottom: 2px solid #e2e8f0;">Qty</th>
                        <th style="padding: 10px 16px; font-size: 12px; color: #64748b; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 700; text-align: right; border-bottom: 2px solid #e2e8f0;">Price</th>
                        <th style="padding: 10px 16px; font-size: 12px; color: #64748b; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 700; text-align: right; border-bottom: 2px solid #e2e8f0;">Subtotal</th>
                    </tr>
                </thead>
                <tbody>
                    {item_rows}
                </tbody>
            </table>

            <!-- Total -->
            <div style="border-top: 2px solid #1e293b; padding-top: 16px; text-align: right; margin-bottom: 32px;">
                <span style="font-size: 13px; color: #6b7280; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600; margin-right: 16px;">Total</span>
                <span style="font-size: 22px; color: #1e293b; font-weight: 800;">{_format_currency(total_price)}</span>
            </div>

            <!-- CTA Note -->
            <div style="background-color: #eff6ff; border-radius: 8px; padding: 16px 20px; text-align: center; margin-bottom: 24px;">
                <p style="margin: 0; font-size: 14px; color: #1e40af; font-weight: 500;">
                    📦 &nbsp;Your order is being prepared. We'll notify you when it's ready for pickup.
                </p>
            </div>

        </div>

        <!-- Footer -->
        <div style="background-color: #f8fafc; padding: 24px 32px; text-align: center; border-top: 1px solid #e2e8f0; border-radius: 0 0 8px 8px;">
            <p style="margin: 0 0 4px; font-size: 14px; color: #374151; font-weight: 600;">ShoeOps Team</p>
            <p style="margin: 0; font-size: 12px; color: #94a3b8;">Inventory &bull; Orders &bull; Intelligence</p>
            <p style="margin: 12px 0 0; font-size: 11px; color: #cbd5e1;">This is an automated notification from ShoeOps Agent.</p>
        </div>

    </div>
    """

In [ ]:
# ============================================================
# REPLACE Cell 29 — build_status_update_email_html (Professional Styled)
# ============================================================

def build_status_update_email_html(order_id: str, customer_name: str, old_status: str, new_status: str, store_id: str) -> str:

    # Dynamic color for the new status badge
    status_colors = {
        "PENDING":    ("background-color: #fef3c7; color: #92400e;", "⏳"),
        "CONFIRMED":  ("background-color: #dbeafe; color: #1e40af;", "✅"),
        "PROCESSING": ("background-color: #e0e7ff; color: #3730a3;", "⚙️"),
        "READY":      ("background-color: #d1fae5; color: #065f46;", "📦"),
        "SHIPPED":    ("background-color: #cffafe; color: #155e75;", "🚚"),
        "DELIVERED":  ("background-color: #dcfce7; color: #166534;", "🎉"),
        "CANCELLED":  ("background-color: #fee2e2; color: #991b1b;", "❌"),
    }
    new_upper = (new_status or "").upper().strip()
    old_upper = (old_status or "").upper().strip()
    badge_style, badge_icon = status_colors.get(new_upper, ("background-color: #f3f4f6; color: #374151;", "📋"))

    return f"""
    <div style="max-width: 600px; margin: 0 auto; background-color: #ffffff; font-family: 'Segoe UI', Arial, sans-serif;">

        <!-- Header Banner -->
        <div style="background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%); padding: 32px 40px; text-align: center; border-radius: 8px 8px 0 0;">
            <div style="font-size: 32px; margin-bottom: 8px;">👟</div>
            <h1 style="margin: 0; font-size: 24px; font-weight: 700; color: #ffffff; letter-spacing: 0.5px;">ShoeOps</h1>
            <p style="margin: 4px 0 0; font-size: 13px; color: #94a3b8; letter-spacing: 1px; text-transform: uppercase;">Order Status Update</p>
        </div>

        <!-- Body -->
        <div style="padding: 28px 32px;">

            <p style="font-size: 15px; color: #374151; line-height: 1.6; margin: 0 0 20px;">
                Hi <strong>{customer_name}</strong>,
            </p>
            <p style="font-size: 15px; color: #374151; line-height: 1.6; margin: 0 0 28px;">
                The status of your order has been updated:
            </p>

            <!-- Status Transition -->
            <div style="text-align: center; margin-bottom: 32px;">
                <table style="margin: 0 auto; border-collapse: collapse;">
                    <tr>
                        <!-- Old Status -->
                        <td style="text-align: center; padding: 0 12px;">
                            <p style="margin: 0 0 6px; font-size: 11px; color: #94a3b8; text-transform: uppercase; letter-spacing: 1px; font-weight: 600;">Previous</p>
                            <span style="display: inline-block; background-color: #f1f5f9; color: #64748b; font-size: 13px; font-weight: 700; padding: 6px 18px; border-radius: 16px; text-transform: uppercase; letter-spacing: 0.5px; text-decoration: line-through;">{old_upper}</span>
                        </td>
                        <!-- Arrow -->
                        <td style="padding: 12px 8px 0; font-size: 22px; color: #94a3b8; vertical-align: bottom;">→</td>
                        <!-- New Status -->
                        <td style="text-align: center; padding: 0 12px;">
                            <p style="margin: 0 0 6px; font-size: 11px; color: #94a3b8; text-transform: uppercase; letter-spacing: 1px; font-weight: 600;">Current</p>
                            <span style="display: inline-block; {badge_style} font-size: 13px; font-weight: 700; padding: 6px 18px; border-radius: 16px; text-transform: uppercase; letter-spacing: 0.5px;">{badge_icon} {new_upper}</span>
                        </td>
                    </tr>
                </table>
            </div>

            <!-- Order Info Card -->
            <div style="background-color: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px 24px; margin-bottom: 28px;">
                <table style="width: 100%; border-collapse: collapse;">
                    <tr>
                        <td style="padding: 8px 0; font-size: 13px; color: #6b7280; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600;">Order ID</td>
                        <td style="padding: 8px 0; font-size: 15px; color: #1e293b; font-weight: 700; text-align: right;">{order_id}</td>
                    </tr>
                    <tr>
                        <td style="padding: 8px 0; font-size: 13px; color: #6b7280; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600;">Store</td>
                        <td style="padding: 8px 0; font-size: 15px; color: #1e293b; font-weight: 600; text-align: right;">{store_id}</td>
                    </tr>
                </table>
            </div>

            <!-- Info Note -->
            <div style="background-color: #eff6ff; border-radius: 8px; padding: 16px 20px; text-align: center; margin-bottom: 24px;">
                <p style="margin: 0; font-size: 14px; color: #1e40af; font-weight: 500;">
                    Questions about your order? Reply to this email or contact your store directly.
                </p>
            </div>

        </div>

        <!-- Footer -->
        <div style="background-color: #f8fafc; padding: 24px 32px; text-align: center; border-top: 1px solid #e2e8f0; border-radius: 0 0 8px 8px;">
            <p style="margin: 0 0 4px; font-size: 14px; color: #374151; font-weight: 600;">ShoeOps Team</p>
            <p style="margin: 0; font-size: 12px; color: #94a3b8;">Inventory &bull; Orders &bull; Intelligence</p>
            <p style="margin: 12px 0 0; font-size: 11px; color: #cbd5e1;">This is an automated notification from ShoeOps Agent.</p>
        </div>

    </div>
    """

In [ ]:
def send_order_created_notification(order_id: str, customer_id: str, customer_name: str, store_id: str, items: List[dict], total_price: float) -> dict:
    contact = get_customer_contact_by_customer_id(customer_id)
    if not contact.get("success"):
        return {"success": False, "error": contact.get("error", "Customer contact lookup failed.")}

    to_email = (contact.get("email") or "").strip()
    if not to_email:
        return {"success": False, "error": f"No email is registered for customer_id {customer_id}."}

    html_body = build_order_created_email_html(
        order_id=order_id,
        customer_name=customer_name,
        store_id=store_id,
        items=items,
        total_price=total_price,
    )

    return send_email_via_composio(
        to_email=to_email,
        to_name=contact.get("display_name") or customer_name,
        subject=f"ShoeOps Order Confirmation — {order_id}",
        html_body=html_body,
    )

In [ ]:
def send_status_update_notification(order_id: str, customer_id: str, customer_name: str, old_status: str, new_status: str, store_id: str) -> dict:
    contact = get_customer_contact_by_customer_id(customer_id)
    if not contact.get("success"):
        return {"success": False, "error": contact.get("error", "Customer contact lookup failed.")}

    to_email = (contact.get("email") or "").strip()
    if not to_email:
        return {"success": False, "error": f"No email is registered for customer_id {customer_id}."}

    html_body = build_status_update_email_html(
        order_id=order_id,
        customer_name=customer_name,
        old_status=old_status,
        new_status=new_status,
        store_id=store_id,
    )

    return send_email_via_composio(
        to_email=to_email,
        to_name=contact.get("display_name") or customer_name,
        subject=f"ShoeOps Order Status Updated — {order_id}",
        html_body=html_body,
    )

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
def fetch_schema_once_manager() -> str:
    """
    Build a compact schema string for Text2SQL.
    IMPORTANT: exclude sensitive tables like users (pin_code).
    """
    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute(
            """
            SELECT table_name, column_name, data_type
            FROM information_schema.columns
            WHERE table_schema='public'
            ORDER BY table_name, ordinal_position
            """
        )
        rows = cur.fetchall()
    finally:
        conn.close()

    # tables to exclude from LLM visibility (security)
    EXCLUDE_TABLES = {"users"}  # add more if needed

    tables: dict[str, list[tuple[str, str]]] = {}
    for table, col, dtype in rows:
        if table in EXCLUDE_TABLES:
            continue
        tables.setdefault(table, []).append((col, dtype))

    # format: TABLE x(col type, col type, ...)
    lines = []
    for t in sorted(tables.keys()):
        cols = ", ".join([f"{c} {d}" for (c, d) in tables[t]])
        lines.append(f"TABLE {t}({cols})")
    return "\n".join(lines)

GLOBAL_SCHEMA_MANAGER = fetch_schema_once_manager()

In [ ]:
# ============================================================
# Loads store directory + product brands once at startup
# and embeds them into prompts so the LLM has business context
# ============================================================

def _fetch_store_directory() -> str:
    """Fetch all stores from DB and format as a lookup table for prompts."""
    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute("SELECT store_id, name, city FROM stores ORDER BY store_id")
        rows = cur.fetchall()
    except Exception:
        return "  (Could not load store directory)"
    finally:
        conn.close()

    if not rows:
        return "  (No stores found)"

    lines = []
    for store_id, name, city in rows:
        lines.append(f"  • {store_id}  →  {name}  (city: {city})")
    return "\n".join(lines)


def _fetch_product_catalog_summary() -> str:
    """Fetch distinct models/brands from DB for prompt context."""
    conn = get_db_conn()
    try:
        cur = conn.cursor()
        # Get brand/model summary with size ranges
        cur.execute("""
            SELECT model, color, MIN(size) AS min_size, MAX(size) AS max_size,
                   COUNT(*) AS variants, MIN(price) AS min_price, MAX(price) AS max_price
            FROM products
            GROUP BY model, color
            ORDER BY model, color
        """)
        rows = cur.fetchall()
    except Exception:
        return "  (Could not load product catalog)"
    finally:
        conn.close()

    if not rows:
        return "  (No products in catalog)"

    lines = []
    for model, color, min_sz, max_sz, variants, min_p, max_p in rows:
        size_range = f"size {min_sz}" if min_sz == max_sz else f"sizes {min_sz}-{max_sz}"
        price_range = f"${float(min_p):.0f}" if min_p == max_p else f"${float(min_p):.0f}-${float(max_p):.0f}"
        lines.append(f"  • {model} / {color} — {size_range} — {price_range} ({variants} SKUs)")
    return "\n".join(lines)


# Build once at startup
STORE_DIRECTORY = _fetch_store_directory()
PRODUCT_CATALOG_SUMMARY = _fetch_product_catalog_summary()

print("✅ Business context loaded:")
print(f"  Store directory:\n{STORE_DIRECTORY}")
print(f"  Product catalog:\n{PRODUCT_CATALOG_SUMMARY}")

✅ Business context loaded:
  Store directory:
  • store_hrz  →  Herzliya Mall  (city: Herzliya)
  • store_jlm  →  Jerusalem Outlet  (city: Jerusalem)
  • store_ntn  →  Netanya Center  (city: Netanya)
  • store_tlv  →  Tel Aviv Flagship  (city: Tel Aviv)
  Product catalog:
  • Air Force 1 / Black — size 42 — $450 (1 SKUs)
  • Air Force 1 / White — sizes 42-43 — $450 (2 SKUs)
  • New Balance 574 / Navy — sizes 42-43 — $400 (2 SKUs)
  • New Balance 574 / Red — size 42 — $400 (1 SKUs)
  • Nike Cortez / white — size 47 — $500 (1 SKUs)
  • Nike Cortez / White — size 44 — $300 (1 SKUs)
  • Puma RS-X / Black — size 42 — $350 (1 SKUs)
  • Test Runner / Blue — size 42 — $300 (1 SKUs)
  • TST / RED — size 43 — $200 (1 SKUs)
  • Ultraboost 5 / Gray — size 41 — $700 (1 SKUs)
  • Ultraboost 5 / White — sizes 42-43 — $700 (2 SKUs)


In [ ]:
def _strip_sql_fences(txt: str) -> str:
    s = (txt or "").strip()
    if s.startswith("```"):
        # remove ```sql ... ```
        s = s.split("```", 2)
        s = s[1] if len(s) > 1 else ""
        s = s.replace("sql", "", 1).strip()
    return s.strip().strip(";").strip()

In [ ]:
@tool
def sql_writer(question: str) -> str:
    """Generate a SQL query (SELECT, INSERT, or UPDATE) for manager operations (uses GLOBAL_SCHEMA_MANAGER)."""
    sys = SystemMessage(content=f"""
You are a Text2SQL generator for a PostgreSQL database.
Schema:
{GLOBAL_SCHEMA_MANAGER}

TABLE RELATIONSHIPS (use these for JOINs):
- orders.customer_id → references a customer (format: cust_XXX)
- orders.store_id → references stores.store_id (can be NULL)
- order_items.order_id → references orders.order_id
- order_items.sku → references products.sku
- inventory.store_id → references stores.store_id
- inventory.sku → references products.sku

IMPORTANT COLUMN LOCATIONS:
- price is on the PRODUCTS table, NOT on order_items
- to compute revenue: JOIN order_items with products on sku, then SUM(order_items.quantity * products.price)
- order dates: orders.created_at is a TIMESTAMP column
- available stock = inventory.on_hand - inventory.reserved

COMMON QUERY PATTERNS:
- Revenue: SELECT SUM(oi.quantity * p.price) FROM order_items oi JOIN products p ON oi.sku = p.sku JOIN orders o ON oi.order_id = o.order_id WHERE ...
- Daily order counts: SELECT DATE(created_at) AS day, COUNT(*) FROM orders GROUP BY DATE(created_at) ORDER BY day
- Top models by qty: SELECT p.model, SUM(oi.quantity) FROM order_items oi JOIN products p ON oi.sku = p.sku GROUP BY p.model ORDER BY SUM(oi.quantity) DESC
- Low availability: SELECT store_id, sku, (on_hand - reserved) AS available FROM inventory WHERE (on_hand - reserved) <= 2

RULES:
- You may generate SELECT, INSERT, or UPDATE queries.
- NEVER generate DELETE, DROP, TRUNCATE, ALTER, CREATE TABLE, or GRANT statements.
- Return ONLY the raw SQL string. No markdown, no explanation, no backticks.
- For INSERT: use the exact column names from the schema.
- For INSERT into inventory: ALWAYS use INSERT ... ON CONFLICT (store_id, sku) DO UPDATE SET ... syntax (UPSERT).
- For UPDATE: always include a WHERE clause.
- Use single quotes for string values.
- If the question asks about columns or data that do NOT exist in the schema (e.g., carrier, delivery_time, shipping_date),
  do NOT generate SQL. Instead return exactly: SCHEMA_LIMITATION: <explain what is missing>
""")
    human = HumanMessage(content=question)
    response = llm.invoke([sys, human])
    return response.content

In [ ]:
@tool
def sql_executor(query: str) -> dict:
    """Execute a safe SELECT query and return rows/columns."""
    q = (query or "").strip()
    q = _strip_sql_fences(q)

    if not q:
        return {"success": False, "error": "Empty SQL query."}

    # Handle SCHEMA_LIMITATION responses from sql_writer
    if q.upper().startswith("SCHEMA_LIMITATION"):
        return {"success": False, "error": q, "schema_limitation": True}

    # block multi-statement
    if ";" in q:
        return {"success": False, "error": "Multiple statements are not allowed."}

    lowered = q.lower()
    banned = ["insert", "update", "delete", "drop", "alter", "create", "truncate", "grant", "revoke"]
    if any(b in lowered for b in banned):
        return {"success": False, "error": "Only SELECT queries are allowed."}

    # block sensitive table explicitly
    if " users" in lowered or "users." in lowered or "from users" in lowered or "join users" in lowered:
        return {"success": False, "error": "Query to 'users' table is not allowed."}

    # must start with SELECT or WITH
    first = lowered.lstrip().split(None, 1)[0] if lowered.strip() else ""
    if first not in ("select", "with"):
        return {"success": False, "error": "Only SELECT (or WITH ... SELECT) is allowed."}

    # enforce LIMIT <= 200 if missing
    if " limit " not in f" {lowered} ":
        q = f"SELECT * FROM ({q}) AS subq LIMIT 200"

    conn = get_db_conn()
    try:
        cur = conn.cursor()
        cur.execute(q)
        rows = cur.fetchall()

        cols = []
        if cur.description:
            cols = [d.name for d in cur.description]

        # safety cap
        if len(rows) > 200:
            rows = rows[:200]

        return {"success": True, "columns": cols, "rows": rows, "row_count": len(rows)}
    except Exception as e:
        return {"success": False, "error": f"DB error: {e}", "query": q}
    finally:
        conn.close()

In [ ]:
class SubmitAnswer(BaseModel):
    """Use for every user-facing answer (NO plain text)."""
    answer: str = Field(description="Final answer shown to the user.")
    source: Literal["GENERAL", "DB", "RAG", "WEB", "DB+RAG", "RAG+WEB", "DB+WEB", "DB+RAG+WEB"] = Field(
        description="Where the answer came from in THIS user turn."
    )

In [ ]:
class SubmitDraftAction(BaseModel):
    """Use when proposing a write operation (requires manager approval)."""
    action_type: Literal["TRANSFER", "STATUS_UPDATE", "DB_WRITE"] = Field(description="Which write action to execute")

    # Transfer fields
    from_store: Optional[str] = None
    to_store: Optional[str] = None
    sku: Optional[str] = None
    qty: Optional[int] = None

    # Status update fields
    order_id: Optional[str] = None
    new_status: Optional[str] = None

    # DB_WRITE fields (inventory update, product create/delete)
    sql_query: Optional[str] = None
    description: Optional[str] = None
    auto_create_inventory: Optional[bool] = Field(default=False, description="If True, after INSERT into products, also create inventory rows (on_hand=0) at all stores.")

    risk_level: Literal["LOW", "MED", "HIGH"] = Field(description="Operational risk level.")
    rationale: str = Field(description="Why this action is needed.")

In [ ]:
# ============================================================
# REPLACE Cell 39 — PlannerDecision (added target_customer_id, target_customer_name)
# ============================================================

class PlannerDecision(BaseModel):
    intent: Literal[
        "product_query",
        "availability_query",
        "order_query",
        "order_list",
        "inventory_query",
        "inventory_update",
        "product_create",
        "product_delete",
        "analytics_query",
        "policy_question",
        "transfer",
        "status_update",
        "register_account",
        "place_order",
        "confirm_order",
        "other",
    ] = Field(description="High-level intent for routing.")

    order_id: Optional[str] = None
    sku: Optional[str] = None
    size: Optional[int] = None
    qty: Optional[int] = None
    from_store: Optional[str] = None
    to_store: Optional[str] = None
    new_status: Optional[str] = None

    store_id: Optional[str] = None

    # Registration fields
    reg_username: Optional[str] = None
    reg_pin_code: Optional[str] = None
    reg_display_name: Optional[str] = None
    reg_email: Optional[str] = None

    # Manager-on-behalf-of-customer ordering
    target_customer_id: Optional[str] = Field(default=None, description="Customer ID when manager places order on behalf of a customer. E.g. 'CUST-001'.")
    target_customer_name: Optional[str] = Field(default=None, description="Customer name/search term when manager places order on behalf of a customer.")

    missing_fields: List[str] = Field(default_factory=list)
    allowed: bool = Field(default=True)
    needs_executor: bool = Field(default=True)
    response_to_user: str = Field(default="")

In [ ]:
class FormatUserReply(BaseModel):
    """Use ONLY for the final user-facing message after deterministic classification."""
    answer: str = Field(description="Clear user-facing response.")
    source: Literal["GENERAL", "DB", "RAG", "WEB", "DB+RAG", "RAG+WEB", "DB+WEB", "DB+RAG+WEB"] = Field(
        description="Source label for this answer."
    )

In [ ]:
# ============================================================
# REPLACE Cell 41 — AgentState (added target_customer_id, target_customer_name)
# ============================================================

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

    user_role: Literal["customer", "manager", "guest"]
    user_display_name: str
    customer_id: Optional[str]

    needs_manager_approval: bool

    intent: Optional[str]
    order_id: Optional[str]
    sku: Optional[str]
    size: Optional[int]
    qty: Optional[int]
    from_store: Optional[str]
    to_store: Optional[str]
    new_status: Optional[str]
    store_id: Optional[str]

    # Registration fields
    reg_username: Optional[str]
    reg_pin_code: Optional[str]
    reg_display_name: Optional[str]
    reg_email: Optional[str]

    # Manager-on-behalf-of-customer ordering
    target_customer_id: Optional[str]
    target_customer_name: Optional[str]

    # Customer order (set by evidence_collector when prepare_order_summary succeeds)
    pending_order: Optional[dict]

    missing_fields: List[str]
    plan: Optional[str]
    draft_action: Optional[dict]
    evidence: Optional[dict]

    manager_decision: Optional[str]
    manager_feedback: Optional[str]

    planner_allowed: bool
    planner_needs_executor: bool
    planner_response: str

    pending_user_reply: Optional[dict]

In [ ]:
def _latest_user_request_text(msgs: List[BaseMessage]) -> str:
    for m in reversed(msgs or []):
        if isinstance(m, HumanMessage):
            return (m.content or "").strip()
    return ""

In [ ]:
def _close_last_submit_draft_if_present(state: AgentState) -> List[ToolMessage]:
    last = state["messages"][-1] if state.get("messages") else None
    tool_calls = getattr(last, "tool_calls", None) or []
    closes = []

    for tc in tool_calls:
        if tc.get("name") == "SubmitDraftAction":
            closes.append(
                ToolMessage(
                    tool_call_id=tc["id"],
                    name="SubmitDraftAction",
                    content="Reviewed.",
                )
            )
    return closes

In [ ]:
FORMATTER_PROMPT = """You are ShoeOps Response Formatter.

Your job:
- Convert a deterministic backend payload into a polished user-facing message.
- You MUST call the FormatUserReply tool.
- Do NOT invent facts.
- Do NOT change the outcome type.
- Be concise, natural, and business-friendly.

Rules:
- If success=true: write a positive confirmation.
- If success=false: explain the real reason clearly.
- Prefer the normalized error_code over raw_error.
- Mention concrete details when present (order_id, sku, qty, store IDs, statuses).
- Never mention internal nodes, tool flow, JSON, stack traces, or system implementation details.
- Never say "please retry" unless retryable=true.
- Keep the answer to 1-2 short sentences max.
"""

In [ ]:
# Block — Brand-to-model mapping (loaded into planner + executor prompts)

BRAND_MODEL_MAP = """
BRAND → MODEL MAPPING (use this to resolve brand names to product models):
- Nike: Air Force 1, Nike Cortez
- Adidas: Ultraboost 5
- Puma: Puma RS-X
- New Balance: New Balance 574

When a user asks for a BRAND (e.g., "Nike shoes", "Adidas sneakers"), you MUST map it
to the correct model names above and search/filter by those models.
Example: "Show me Nike shoes" → search for model="Air Force 1" AND model="Nike Cortez".
"""

In [ ]:
# ============================================================
# REPLACE Cell 45 — PLANNER_PROMPT (with brand mapping + web search guidance)
# ============================================================

PLANNER_PROMPT = f"""You are ShoeOps Planner — the routing brain of a shoe retail operations agent.

=== ABOUT SHOEOPS ===
ShoeOps is a multi-store shoe retail chain. The system manages inventory, orders, transfers,
and customer service across physical stores. Here is the current business data:

STORE DIRECTORY (use these store_id values when routing):
{STORE_DIRECTORY}

PRODUCT CATALOG:
{PRODUCT_CATALOG_SUMMARY}

{BRAND_MODEL_MAP}

IMPORTANT — STORE NAME RESOLUTION:
Users often refer to stores by city name, nickname, or abbreviation. You MUST map them:
- "tel aviv" / "tlv" / "t.a." / "ta store" → look for the store whose city contains "Tel Aviv"
- "jerusalem" / "jlm" / "j-m" → look for the store whose city contains "Jerusalem"
- "haifa" → look for the store whose city contains "Haifa"
- If unsure, set needs_executor=true so the executor can call query_stores to resolve it.
- NEVER say "store not found" just because the user used a city name instead of a store_id.
  Instead, match the city name to the correct store_id from the directory above.

RULES (STRICT):
- You MUST call PlannerDecision tool. No plain text.
- You must NOT call any DB/RAG/WEB tools.
- Your job: classify intent, extract fields, and resolve store_id from city names. Nothing else.
- Always set allowed=true and needs_executor=true. Permission checks are handled elsewhere.

=== INTENT CLASSIFICATION ===
- product_query: asking about products, models, colors, sizes, prices, "what do you sell", "show me shoes".
    This ALSO covers brand-based questions like "show me Nike shoes" or "what Adidas do you have".
    This ALSO covers general shoe knowledge questions like "latest Nike releases", "best running shoes",
    "tell me about Air Force 1", shoe comparisons, brand info, trends, recommendations.
    The executor has BOTH a local product database AND a web search tool (tavily_search) to answer these.
- availability_query: asking what's available. This covers TWO directions:
    (a) which stores have a specific product in stock (e.g., "where can I find Air Force 1?")
    (b) what products are available at a specific store (e.g., "what's available in Tel Aviv?", "what does the TLV store have?", "show me shoes at Haifa store")
    IMPORTANT: if a customer or guest asks about a store's products, this is availability_query, NOT inventory_query.
    Set store_id when the user mentions a store/city. Set sku when the user mentions a product.
- order_query: asking about a specific order (needs order_id)
- order_list: asking to see their orders / order history
- inventory_query: asking about detailed stock LEVELS — on_hand, reserved, internal stock counts. This is manager-only operational data. Examples: "check inventory levels", "how much stock do we have", "show me the inventory report".
    IMPORTANT: "what's available at store X" is availability_query, NOT inventory_query.
- inventory_update: requesting to change stock quantities (add stock, set on_hand)
- product_create: requesting to add/create a new product to the catalog
- product_delete: requesting to remove/discontinue a product
- analytics_query: asking for reports, statistics, totals, summaries, comparisons
- policy_question: asking about policies, shipping, returns, SOP, procedures
- transfer: requesting to move stock between stores
- status_update: requesting to change an order status
- register_account: user wants to create a new account
- place_order: customer OR manager wants to order items for store pickup
- confirm_order: customer OR manager says "confirm" / "yes" / "place it" AFTER seeing an order summary
- other: general greetings, "what can you do", anything that doesn't fit above

=== REQUIRED FIELDS BY INTENT ===
- transfer: from_store, to_store, sku, qty
- status_update: order_id, new_status
- availability_query: sku OR store_id (at least one). If user says a product name/model, set sku="" and let executor resolve it. If user says a store/city, set store_id.
- order_query: order_id
- order_list: no required fields
- inventory_query: NO required fields. The executor can query all stores, one store, or one SKU.
  → If user says "check stock for tel aviv" → set store_id to the matching store_id from directory.
  → If user says "check all stock" or "show me everything" → leave store_id and sku empty.
- inventory_update: sku, store_id, qty
- product_create: sku (or model+color+size if user describes the product naturally)
- product_delete: sku
- analytics_query: no required fields
- policy_question: no required fields
- register_account: reg_username, reg_pin_code, reg_display_name, reg_email
  → If ANY are missing, put ALL FOUR in missing_fields.
- place_order:
  → For CUSTOMERS: store_id is required. Items are extracted by executor.
  → For MANAGERS placing on behalf of a customer:
    - If the manager mentions a customer name, username, email, or ID → put it in target_customer_name
      (or target_customer_id if it looks like a CUST-XXX ID). The executor will resolve it.
    - DO NOT add "target_customer_id" to missing_fields if ANY customer reference was provided.
      The executor has lookup_customer to resolve names to IDs.
    - ONLY add "target_customer_id" to missing_fields if the manager mentioned NO customer at all
      (e.g., just said "place an order" without saying who for).
    - store_id is also needed. If missing, add "store_id" to missing_fields.
- confirm_order: no required fields (pending_order is in state)
- other: no required fields

=== HINTS ===
- "in which stores" / "which branch has" → availability_query
- "what's available at" / "what does store X have" / "show me what's in TLV" → availability_query
- "check stock levels" / "show inventory report" / "on_hand quantities" → inventory_query
- "create an account" / "register" / "sign up" → register_account
- "add a new product" / "create product" → product_create
- "update stock" / "set on_hand" / "add X units" → inventory_update
- "delete product" / "remove product" / "discontinue" → product_delete
- "I want to buy" / "order" / "purchase" / "add to cart" → place_order
- "place an order for customer X" / "order for CUST-001" / "order for Alon" → place_order (manager on behalf)
- "confirm" / "yes place it" / "go ahead" (after an order summary was shown) → confirm_order
- If user says "confirm" but there was no prior order summary, classify as "other"
- "what do you sell" / "what brands" / "show me shoes" → product_query
- "Nike shoes" / "Adidas sneakers" / "show me Puma" → product_query (use brand mapping above)
- "latest Nike releases" / "best running shoes 2025" / "shoe trends" → product_query (executor will use web search)
- "how many orders" / "total revenue" / "top selling" → analytics_query
- "what can you do" / "help" → other (respond with capabilities summary)
"""

In [ ]:
# ============================================================
# REPLACE Cell 46 — EXECUTOR_PROMPT (with brand mapping + tavily web search guidance)
# ============================================================

EXECUTOR_PROMPT = f"""You are ShoeOps Executor — the action engine of a shoe retail operations agent.

=== ABOUT SHOEOPS ===
ShoeOps is a multi-store shoe retail chain. You help manage inventory, orders, transfers,
and customer service. Here is the current business data:

STORE DIRECTORY:
{STORE_DIRECTORY}

PRODUCT CATALOG:
{PRODUCT_CATALOG_SUMMARY}

{BRAND_MODEL_MAP}

=== STORE NAME RESOLUTION ===
When the user (or planner) refers to a store by city name, nickname, or abbreviation,
ALWAYS resolve it to the correct store_id from the directory above. Examples:
- "tel aviv" / "tlv" / "TA" → use the store_id whose city is Tel Aviv
- "jerusalem" / "jlm" → use the store_id whose city is Jerusalem
If you're unsure which store the user means, call query_stores(city=...) to look it up.
NEVER tell the user "store not found" when they used a valid city name.

RULES (STRICT):
- You MUST respond using a TOOL CALL: either SubmitAnswer (normal) or SubmitDraftAction (write proposal). No plain text.
- You may call READ tools (DB/RAG/WEB) as needed.
- You MUST NOT call write tools directly (execute_transfer/update_order_status/log_audit).
- If you need more info, ask via SubmitAnswer.
- NEVER generate or propose DELETE SQL.

=== WEB SEARCH GUIDANCE ===
You have access to tavily_search(query=...) for real-time web information. USE IT PROACTIVELY for:
- Brand questions beyond the local catalog ("latest Nike releases", "new Adidas models 2025")
- Shoe recommendations, comparisons, reviews ("best running shoes", "Air Force 1 vs Ultraboost")
- Trends and news ("shoe trends 2025", "most popular sneakers")
- Any product knowledge question the local DB cannot answer
- When a user asks about a brand or model you don't recognize
Do NOT say "I don't have that information" — search the web first, then answer.
Combine web results with local catalog info when relevant (e.g., "We carry Air Force 1 — here's what Nike says about the latest colorways: [web results]").

=== TOOL GUIDANCE BY INTENT ===

- intent=product_query:
  STEP 1: Check the BRAND → MODEL MAPPING above. If the user asked for a brand (e.g., "Nike"),
    map it to the correct model names and call query_products for EACH matching model.
    Example: "Nike shoes" → call query_products(model="Air Force 1") AND query_products(model="Nike Cortez").
  STEP 2: If the user asked about products NOT in our catalog, or asked about new releases,
    trends, comparisons, or general shoe knowledge → call tavily_search(query=...) to get web info.
  STEP 3: Combine local catalog results with web search results into a helpful SubmitAnswer.
  If the user asked a simple catalog question (e.g., "what do you sell") → just query_products() is enough.

- intent=availability_query:
  This tool works in two directions:
  (a) By SKU: call query_store_availability(sku=...) → which stores have it.
  (b) By store: call query_store_availability(store_id=...) → what products are available there.
  (c) Both: call query_store_availability(sku=..., store_id=...) → specific check.
  If the user described a product by name (e.g., "Air Force 1 white size 42"), first call
  query_products to find the matching SKU, then call query_store_availability with that SKU.
  If the user asked about a store (e.g., "what's at TLV?"), resolve the city to a store_id
  using the store directory and call query_store_availability(store_id=...).

- intent=inventory_query:
  call query_inventory with whatever filters are known.
  - If store_id is known → query_inventory(store_id=...) — shows all products at that store.
  - If sku is known → query_inventory(sku=...) — shows that product across all stores.
  - If BOTH are known → query_inventory(store_id=..., sku=...).
  - If NEITHER is known → query_inventory() with no arguments — shows everything (up to 200 rows).
  IMPORTANT: query_inventory WORKS without any parameters. Do NOT tell the user you need a store_id.

- intent=order_list:
  call list_my_orders(customer_id=...).

- intent=order_query:
  if user_role=customer: call query_order_customer(order_id=..., customer_id=...).
  if user_role=manager: call query_order_manager(order_id=...).

- intent=policy_question:
  call search_policy(query=...) BEFORE answering. Always ground your answer in the RAG results.

- intent=analytics_query (manager only):
  1) call sql_writer(question=...)
  2) call sql_executor(query=...)
  3) then call SubmitAnswer with a clear summary of the data.

- intent=register_account (guest only):
  1) call check_username_exists(username=<reg_username>).
  2) If exists → SubmitAnswer saying username is taken.
  3) If free → call register_new_user(username=..., pin_code=..., display_name=..., email=...).
  4) SubmitAnswer with welcome message or error.

- intent=inventory_update (manager only — DB_WRITE):
  1) Resolve the store_id from the store directory if the user used a city name.
  2) Call sql_writer with UPSERT pattern.
  3) Call SubmitDraftAction(action_type="DB_WRITE", sql_query=..., description=..., risk_level="MED").

- intent=product_create (manager only — DB_WRITE):
  1) Call sql_writer(INSERT INTO products ...).
  2) Call SubmitDraftAction(action_type="DB_WRITE", sql_query=..., auto_create_inventory=true, sku=..., risk_level="MED").

- intent=product_delete (manager only — DB_WRITE):
  1) Call sql_writer(UPDATE inventory SET on_hand=0, reserved=0 WHERE sku=...).
  2) Call SubmitDraftAction(action_type="DB_WRITE", sql_query=..., risk_level="HIGH").

- intent=transfer (manager only):
  Resolve store names to store_ids using the directory above.
  Propose via SubmitDraftAction(action_type="TRANSFER", from_store=..., to_store=..., sku=..., qty=..., risk_level=..., rationale=...).

- intent=status_update (manager only):
  Propose via SubmitDraftAction(action_type="STATUS_UPDATE", order_id=..., new_status=..., risk_level=..., rationale=...).

- intent=place_order:
  This is a TWO-STEP process. In this step, you prepare the summary.

  **IF user_role=customer:**
    STEP 0: Check if store_id is in the StateContext.
      - If store_id is MISSING or empty:
        a) First use query_products to find the matching SKU(s).
           If user mentions a brand, use the BRAND → MODEL MAPPING to find the right models.
        b) Then call query_store_availability(sku=...) to find ALL stores that have it in stock.
        c) Call SubmitAnswer listing ALL available stores with their available quantities,
           and ask the customer to choose which store they want to order from.
           Example: "Air Force 1 Black size 42 is available at:\n- Herzliya Mall (7 in stock)\n- Netanya Branch (10 in stock)\nWhich store would you like to order from?"
        d) Do NOT pick a store for the customer. Do NOT call prepare_order_summary yet.
    STEP 1: Once store_id IS known, use query_products to find the SKUs matching what the customer wants.
    STEP 2: Once you have all SKUs and quantities, call prepare_order_summary with:
      items_json = JSON string of [{{"sku": "...", "qty": 1}}, ...]
      store_id = the store from the state context
    STEP 3: If all_available=True → call SubmitAnswer with a formatted order summary and ask to confirm.
            If some items unavailable → call SubmitAnswer listing out-of-stock items.

  **IF user_role=manager (placing order on behalf of a customer):**
    STEP 0: Resolve the target customer.
      - If target_customer_id is in the StateContext → use it directly.
      - If target_customer_name is in the StateContext but no ID → call lookup_customer(search_term=<n>) to find their customer_id.
      - If lookup returns multiple matches → call SubmitAnswer listing the matches and ask the manager to specify which one.
      - If lookup returns no match → call SubmitAnswer saying customer not found.
    STEP 0.5: Check if store_id is in the StateContext.
      - If store_id is MISSING → call query_store_availability(sku=...) to list available stores,
        then call SubmitAnswer listing them and ask the manager to specify. Do NOT pick a store automatically.
    STEP 1: Once store_id IS known, use query_products to find the matching SKUs.
    STEP 2: Call prepare_order_summary with items_json and store_id.
    STEP 3: If all_available=True → call SubmitAnswer with the order summary, clearly stating
            "Order on behalf of: <customer_name> (<customer_id>)" and ask the manager to confirm.
            If some items unavailable → call SubmitAnswer with the stock issue.

- intent=confirm_order:
  confirm_order is handled deterministically by the planner. You should NOT see this intent.
  If you do, call SubmitAnswer("Please say 'confirm' to confirm your pending order.").

- intent=other:
  For greetings: respond warmly, introduce yourself as the ShoeOps Agent. Mention you can help with
  browsing products, checking stock, placing orders, policy questions, and (for managers) inventory
  management, transfers, order status updates, and placing orders on behalf of customers.
  For "what can you do": list your capabilities based on the user's role.

IMPORTANT:
- Use the structured state context (intent + known fields).
- For DB_WRITE: ALWAYS use sql_writer first.
- For place_order: ALWAYS use query_products first to resolve SKUs, then prepare_order_summary.
  NEVER call execute_customer_order directly. Order execution is handled by the system after the customer confirms.
- When presenting inventory/product data, format it clearly with model names, not just SKUs.
- For manager orders: the customer_id passed to execute_customer_order must be the TARGET customer's ID, never the manager's.
- When you don't know something about shoes or brands, USE tavily_search before saying "I don't know".
"""

In [ ]:
# ============================================================
# REPLACE Cell 47 — Toolsets (manager gets order tools + lookup_customer)
# execute_customer_order REMOVED from all lists — handled deterministically
# ============================================================

# Block R5 — Toolsets
READ_TOOLS_CUSTOMER = [
    query_stores,
    query_products,
    query_store_availability,
    list_my_orders,
    query_order_customer,
    search_policy,
    tavily_search,
    prepare_order_summary,
    # execute_customer_order — REMOVED: handled deterministically in planner_apply_node
]

READ_TOOLS_MANAGER = [
    query_stores,
    query_products,
    query_order_manager,
    query_inventory,
    search_policy,
    tavily_search,
    sql_writer,
    sql_executor,
    lookup_customer,
    prepare_order_summary,
    # execute_customer_order — REMOVED: handled deterministically in planner_apply_node
]

READ_TOOLS_GUEST = [
    query_stores,
    query_products,
    query_store_availability,
    search_policy,
    tavily_search,
    check_username_exists,
    register_new_user,
]

READ_TOOL_NODE_ALL = [
    query_stores,
    query_products,
    query_store_availability,
    list_my_orders,
    query_order_customer,
    query_order_manager,
    query_inventory,
    search_policy,
    tavily_search,
    sql_writer,
    sql_executor,
    check_username_exists,
    register_new_user,
    lookup_customer,
    prepare_order_summary,
    # execute_customer_order — REMOVED: handled deterministically in planner_apply_node
]
READ_TOOL_NAMES = {t.name for t in READ_TOOL_NODE_ALL}

SIGNALING_TOOLS = [SubmitAnswer, SubmitDraftAction]

In [ ]:
# Block 6E — Helper: pick tools for THIS turn (role-based)
def _read_toolset_for_state(state: AgentState):
    role = state.get("user_role", "customer")
    if role == "manager":
        return list(READ_TOOLS_MANAGER)
    if role == "guest":
        return list(READ_TOOLS_GUEST)
    return list(READ_TOOLS_CUSTOMER)

In [ ]:
# Block 7 — REPLACE formatter_node with this full function
def formatter_node(state: AgentState) -> Command[Literal["finalize"]]:
    payload = dict(state.get("pending_user_reply") or {})
    last_user_text = _latest_user_request_text(state.get("messages", []) or [])

    if not payload:
        synthetic = AIMessage(
            content="",
            tool_calls=[{
                "name": "SubmitAnswer",
                "id": str(uuid4()),
                "type": "tool_call",
                "args": {
                    "answer": "❌ The request could not be completed.",
                    "source": "DB",
                },
            }],
        )
        return Command(
            update={
                "messages": [synthetic],
                "pending_user_reply": None,
            },
            goto="finalize",
        )

    base_text = payload.get("fallback_text") or (
        "✅ Done." if payload.get("success") else "❌ The request could not be completed."
    )
    warning_text = (payload.get("warning_text") or "").strip()
    deterministic_text = base_text + (f"\n\n⚠️ {warning_text}" if warning_text else "")

    # If there is a deterministic warning, bypass the LLM formatter so the warning is guaranteed to appear.
    if warning_text:
        synthetic = AIMessage(
            content="",
            tool_calls=[{
                "name": "SubmitAnswer",
                "id": str(uuid4()),
                "type": "tool_call",
                "args": {
                    "answer": deterministic_text,
                    "source": payload.get("source", "DB"),
                },
            }],
        )
        return Command(
            update={
                "messages": [synthetic],
                "pending_user_reply": None,
            },
            goto="finalize",
        )

    llm_bound = llm.bind_tools([FormatUserReply])

    sys = SystemMessage(content=FORMATTER_PROMPT)
    human = HumanMessage(content=json.dumps({
        "last_user_request": last_user_text,
        "payload": payload,
    }, ensure_ascii=False))

    resp = llm_bound.invoke([sys, human])

    tool_calls = getattr(resp, "tool_calls", None) or []
    fmt_call = next((tc for tc in tool_calls if tc.get("name") == "FormatUserReply"), None)

    if not fmt_call:
        synthetic = AIMessage(
            content="",
            tool_calls=[{
                "name": "SubmitAnswer",
                "id": str(uuid4()),
                "type": "tool_call",
                "args": {
                    "answer": deterministic_text,
                    "source": payload.get("source", "DB"),
                },
            }],
        )
    else:
        args = fmt_call.get("args", {}) or {}
        synthetic = AIMessage(
            content="",
            tool_calls=[{
                "name": "SubmitAnswer",
                "id": str(uuid4()),
                "type": "tool_call",
                "args": {
                    "answer": args.get("answer", ""),
                    "source": args.get("source", payload.get("source", "DB")),
                },
            }],
        )

    return Command(
        update={
            "messages": [synthetic],
            "pending_user_reply": None,
        },
        goto="finalize",
    )

In [ ]:
# ============================================================
# REPLACE Cell 50 — planner_node (added target_customer fields to context)
# ============================================================

def planner_node(state: AgentState) -> dict:
    llm_bound = llm.bind_tools([PlannerDecision])

    role = state.get("user_role", "customer")
    name = state.get("user_display_name", "User")

    ctx = {
        "intent": state.get("intent"),
        "order_id": state.get("order_id"),
        "sku": state.get("sku"),
        "size": state.get("size"),
        "qty": state.get("qty"),
        "from_store": state.get("from_store"),
        "to_store": state.get("to_store"),
        "new_status": state.get("new_status"),
        "store_id": state.get("store_id"),
        "missing_fields": state.get("missing_fields", []),
        "needs_manager_approval": state.get("needs_manager_approval", False),
        "manager_feedback": state.get("manager_feedback"),
        "evidence": state.get("evidence") or {},
        "pending_order": state.get("pending_order"),
        # NEW: target customer for manager-on-behalf ordering
        "target_customer_id": state.get("target_customer_id"),
        "target_customer_name": state.get("target_customer_name"),
    }

    sys = SystemMessage(content=PLANNER_PROMPT + f"\n\nSession:\n- user_role={role}\n- user_display_name={name}\n\nCurrentContext:\n{json.dumps(ctx, ensure_ascii=False)}\n")

    raw_msgs = [m for m in state["messages"] if not isinstance(m, SystemMessage)]
    clean_msgs = _sanitize_messages(raw_msgs)
    msgs = [sys] + clean_msgs

    resp = llm_bound.invoke(msgs)
    return {"messages": [resp]}

In [ ]:
# ============================================================
# REPLACE Cell 51 — planner_apply_node (deterministic confirm_order + store selection)
# ============================================================

_ROLE_PERMISSIONS = {
    "guest": {
        "product_query", "availability_query", "policy_question",
        "register_account", "other",
    },
    "customer": {
        "product_query", "availability_query", "order_query", "order_list",
        "policy_question",
        "place_order", "confirm_order",
        "other",
    },
    "manager": {
        "product_query", "availability_query", "order_query", "order_list",
        "inventory_query", "analytics_query", "policy_question",
        "transfer", "status_update",
        "inventory_update", "product_create", "product_delete",
        "place_order", "confirm_order",
        "other",
    },
}

_DENIAL_MESSAGES = {
    "guest_order": "I can't access order information for guest users. Would you like to create an account? Just say 'I want to register'.",
    "guest_manager_action": "That action is only available to managers. As a guest, you can browse products, check availability, and ask about policies. You can also create a customer account by saying 'I want to register'.",
    "guest_place_order": "To place an order, you need a customer account. You can create one by saying 'I want to register'.",
    "customer_register": "You already have an account! You're logged in and ready to go.",
    "customer_manager_action": "That action is only available to managers. As a customer, you can browse products, check availability, view your orders, and place new orders.",
    "manager_register": "You already have a manager account!",
    "default": "Sorry, that action is not available for your current role.",
}

def _get_denial_message(role: str, intent: str) -> str:
    if role == "guest":
        if intent in ("order_query", "order_list"):
            return _DENIAL_MESSAGES["guest_order"]
        if intent in ("place_order", "confirm_order"):
            return _DENIAL_MESSAGES["guest_place_order"]
        return _DENIAL_MESSAGES["guest_manager_action"]
    if role == "customer":
        if intent == "register_account":
            return _DENIAL_MESSAGES["customer_register"]
        return _DENIAL_MESSAGES["customer_manager_action"]
    if role == "manager":
        if intent == "register_account":
            return _DENIAL_MESSAGES["manager_register"]
    return _DENIAL_MESSAGES["default"]


def _handle_place_order_no_store(state: dict, args: dict, close_msg) -> Optional[dict]:
    """When place_order has no store_id, look up product availability and ask user to choose.
    Returns a state update dict if handled, or None to fall through to executor."""

    # Check if planner provided a store or if one is already in state
    planner_store = (args.get("store_id") or "").strip()
    if planner_store:
        return None  # store was provided, let executor handle it

    # Try to figure out what product the user wants from planner args
    sku = (args.get("sku") or "").strip()
    model = (args.get("model") or "").strip()
    color = (args.get("color") or "").strip()
    size = args.get("size")

    # Resolve SKU from product details if not directly given
    resolved_skus = []
    if sku:
        resolved_skus = [sku]
    else:
        # Query products to find matching SKUs
        filters = {}
        if model:
            filters["model"] = model
        if color:
            filters["color"] = color
        if size is not None:
            filters["size"] = size

        if filters:
            try:
                result = query_products.invoke(filters)
                if result.get("success") and result.get("products"):
                    resolved_skus = [p["sku"] for p in result["products"]]
            except Exception:
                pass

    if not resolved_skus:
        # Can't resolve product — let executor handle it (it will ask what they want)
        return None

    # Look up availability across all stores for each SKU
    lines = []
    for s in resolved_skus:
        try:
            avail = query_store_availability.invoke({"sku": s})
            if avail.get("success") and avail.get("items"):
                # Get product info from first item
                first = avail["items"][0]
                product_label = f"{first.get('model', '')} {first.get('color', '')} size {first.get('size', '')}".strip()
                lines.append(f"**{product_label}** (SKU: {s}) is available at:")
                for item in avail["items"]:
                    sid = item["store_id"]
                    qty = item["available"]
                    # Resolve store name from directory
                    store_label = sid
                    try:
                        stores_result = query_stores.invoke({})
                        if stores_result.get("success"):
                            for st in stores_result["stores"]:
                                if st["store_id"] == sid:
                                    store_label = f"{st.get('name', sid)} ({st.get('city', '')})"
                                    break
                    except Exception:
                        pass
                    lines.append(f"- {store_label}: **{qty} in stock**")
            else:
                lines.append(f"SKU {s} is currently **out of stock** at all stores.")
        except Exception:
            pass

    if not lines:
        return None  # couldn't get availability, let executor handle

    lines.append("\nWhich store would you like to order from?")

    return {
        "messages": [close_msg],
        "intent": "place_order",
        "sku": resolved_skus[0] if len(resolved_skus) == 1 else None,
        "store_id": None,
        "planner_allowed": True,
        "planner_needs_executor": False,
        "planner_response": "\n".join(lines),
        "missing_fields": ["store_id"],
    }


def _handle_confirm_order_deterministic(state: dict, close_msg) -> Optional[dict]:
    """Deterministic confirm_order: executes pending_order without LLM involvement.
    Returns a state update dict if handled, or None if it should fall through to executor."""

    pending = state.get("pending_order")
    role = state.get("user_role", "customer")

    # No pending order → deny
    if not pending or not pending.get("items"):
        return {
            "messages": [close_msg],
            "intent": "confirm_order",
            "planner_allowed": True,
            "planner_needs_executor": False,
            "planner_response": "There's no pending order to confirm. Please browse products and start a new order first.",
            "missing_fields": [],
        }

    # Resolve customer_id and customer_name
    if role == "manager":
        cust_id = state.get("target_customer_id")
        cust_name = state.get("target_customer_name")
        if not cust_id:
            return {
                "messages": [close_msg],
                "intent": "confirm_order",
                "planner_allowed": True,
                "planner_needs_executor": False,
                "planner_response": "Which customer is this order for? Please specify the customer name or ID.",
                "missing_fields": ["target_customer_id"],
            }
    else:
        cust_id = state.get("customer_id")
        cust_name = state.get("user_display_name", "Customer")

    if not cust_id:
        return {
            "messages": [close_msg],
            "intent": "confirm_order",
            "planner_allowed": True,
            "planner_needs_executor": False,
            "planner_response": "I can't place the order because your customer ID is missing. Please log out and log back in.",
            "missing_fields": ["customer_id"],
        }

    # Build items_json from pending_order
    store_id = pending.get("store_id", "")
    items = []
    for item in pending.get("items", []):
        if item.get("in_stock"):
            items.append({"sku": item["sku"], "qty": item["qty"]})

    if not items:
        return {
            "messages": [close_msg],
            "intent": "confirm_order",
            "planner_allowed": True,
            "planner_needs_executor": False,
            "planner_response": "None of the items in your pending order are available. Please start a new order.",
            "missing_fields": [],
            "pending_order": None,
        }

    # Execute the order deterministically
    result = execute_customer_order.invoke({
        "items_json": json.dumps(items),
        "store_id": store_id,
        "customer_id": cust_id,
        "customer_name": cust_name,
    })

    if result.get("success"):
        order_id = result.get("order_id", "???")
        total = result.get("total_price", 0)

        # Build item lines using pending_order data (has model/color/size)
        # since execute_customer_order only returns sku/qty/unit_price
        pending_items = pending.get("items", [])
        pending_lookup = {pi["sku"]: pi for pi in pending_items}

        lines = []
        if role == "manager":
            lines.append(f"✅ Order placed successfully on behalf of {cust_name} ({cust_id}):")
        else:
            lines.append("✅ Your order has been placed successfully!")
        lines.append(f"**Order ID:** {order_id}")

        for item in items:
            sku = item["sku"]
            qty = item["qty"]
            pi = pending_lookup.get(sku, {})
            model = pi.get("model", "")
            color = pi.get("color", "")
            size = pi.get("size", "")
            price = pi.get("unit_price", 0)
            lines.append(f"- {qty}x {sku} ({model} {color} size {size}) — ${float(price):,.2f} each")

        lines.append(f"**Total Price:** ${float(total):,.2f}")
        lines.append(f"**Status:** PENDING")

        # Email status
        if result.get("email_sent"):
            lines.append(f"\n📧 A confirmation email has been sent to you.")
        elif result.get("email_warning"):
            lines.append(f"\n⚠️ {result['email_warning']}")

        return {
            "messages": [close_msg],
            "intent": "confirm_order",
            "planner_allowed": True,
            "planner_needs_executor": False,
            "planner_response": "\n".join(lines),
            "missing_fields": [],
            "pending_order": None,
        }
    else:
        error = result.get("error", "Order execution failed.")
        return {
            "messages": [close_msg],
            "intent": "confirm_order",
            "planner_allowed": True,
            "planner_needs_executor": False,
            "planner_response": f"❌ Order failed: {error}",
            "missing_fields": [],
        }


def planner_apply_node(state: AgentState) -> dict:
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", None) or []
    tc = next((x for x in tool_calls if x["name"] == "PlannerDecision"), None)

    if not tc:
        return {
            "messages": [HumanMessage(content="SYSTEM: Planner must call PlannerDecision tool.")],
            "planner_allowed": True,
            "planner_needs_executor": True,
            "planner_response": "Planner error. Please retry.",
            "missing_fields": [],
        }

    args = tc.get("args", {}) or {}
    close = ToolMessage(tool_call_id=tc["id"], name="PlannerDecision", content="PlannerDecision received.")

    def _keep(old, new):
        return old if new in (None, "", []) else new

    new_intent = args.get("intent")
    old_intent = state.get("intent")
    missing_fields = args.get("missing_fields", []) or []

    # --- DETERMINISTIC PERMISSION CHECK ---
    role = state.get("user_role", "customer")
    allowed_intents = _ROLE_PERMISSIONS.get(role, _ROLE_PERMISSIONS["customer"])

    if new_intent and new_intent not in allowed_intents:
        denial_msg = _get_denial_message(role, new_intent)
        return {
            "messages": [close],
            "intent": new_intent,
            "planner_allowed": False,
            "planner_needs_executor": False,
            "planner_response": denial_msg,
            "missing_fields": [],
        }

    # --- DETERMINISTIC CONFIRM_ORDER (no LLM involvement) ---
    if new_intent == "confirm_order":
        result = _handle_confirm_order_deterministic(state, close)
        if result is not None:
            return result

    # --- DETERMINISTIC PLACE_ORDER: no store → list availability and ask ---
    if new_intent == "place_order":
        result = _handle_place_order_no_store(state, args, close)
        if result is not None:
            return result

    # --- PERMISSION PASSED ---
    needs_exec = True

    if new_intent == "policy_question":
        needs_exec = True
    if new_intent == "register_account":
        needs_exec = True

    # WIPE EVIDENCE + WRITE-FLOW LEFTOVERS IF INTENT CHANGES
    new_evidence = state.get("evidence") or {}
    wipe_write_flow = False
    if new_intent and old_intent and new_intent != old_intent:
        new_evidence = {}
        wipe_write_flow = True

    update = {
        "messages": [close],
        "intent": new_intent,
        "order_id": _keep(state.get("order_id"), args.get("order_id")),
        "sku": _keep(state.get("sku"), args.get("sku")),
        "size": _keep(state.get("size"), args.get("size")),
        "qty": _keep(state.get("qty"), args.get("qty")),
        "from_store": _keep(state.get("from_store"), args.get("from_store")),
        "to_store": _keep(state.get("to_store"), args.get("to_store")),
        "new_status": _keep(state.get("new_status"), args.get("new_status")),
        "store_id": _keep(state.get("store_id"), args.get("store_id")),
        "reg_username": _keep(state.get("reg_username"), args.get("reg_username")),
        "reg_pin_code": _keep(state.get("reg_pin_code"), args.get("reg_pin_code")),
        "reg_display_name": _keep(state.get("reg_display_name"), args.get("reg_display_name")),
        "reg_email": _keep(state.get("reg_email"), args.get("reg_email")),
        "target_customer_id": _keep(state.get("target_customer_id"), args.get("target_customer_id")),
        "target_customer_name": _keep(state.get("target_customer_name"), args.get("target_customer_name")),
        "missing_fields": missing_fields,
        "planner_allowed": True,
        "planner_needs_executor": needs_exec,
        "planner_response": (args.get("response_to_user") or "").strip(),
        "evidence": new_evidence,
    }

    # Clear pending_order if intent moves AWAY from ordering
    if new_intent and new_intent not in ("place_order", "confirm_order"):
        update["pending_order"] = None

    # Clear target customer if intent moves away from ordering
    if new_intent and new_intent not in ("place_order", "confirm_order"):
        update["target_customer_id"] = None
        update["target_customer_name"] = None

    if wipe_write_flow:
        update.update({
            "draft_action": None,
            "needs_manager_approval": False,
            "manager_decision": None,
            "manager_feedback": None,
            "plan": None,
        })

    return update

In [ ]:
def route_after_planner(state: AgentState) -> Literal["ask_user", "executor"]:
    if not state.get("planner_allowed", True):
        return "ask_user"
    if state.get("missing_fields"):
        return "ask_user"
    if not state.get("planner_needs_executor", True):
        return "ask_user"
    return "executor"

In [ ]:
def ask_user_node(state: AgentState) -> dict:
    msg = (state.get("planner_response") or "").strip()
    if not msg:
        mf = state.get("missing_fields", []) or []
        msg = "Please provide: " + (", ".join(mf) if mf else "more details.")

    synthetic = AIMessage(content="", tool_calls=[{
        "name": "SubmitAnswer",
        "id": str(uuid4()),
        "type": "tool_call",
        "args": {"answer": msg, "source": "GENERAL"},
    }])
    return {"messages": [synthetic]}

In [ ]:
# ============================================================
# REPLACE Cell 54 — executor_node (added target_customer fields to context)
# NOTE: _sanitize_messages is unchanged, included here for completeness
# ============================================================

def _sanitize_messages(msgs: List[BaseMessage]) -> List[BaseMessage]:
    """Ensure every AIMessage with tool_calls has matching ToolMessages.
    Insert closers immediately after the AIMessage, not at the end."""

    # First pass: collect all tool_call_ids that have ToolMessage responses
    responded_ids = set()
    for m in msgs:
        if isinstance(m, ToolMessage):
            tid = getattr(m, "tool_call_id", None)
            if tid:
                responded_ids.add(tid)

    # Second pass: rebuild message list, inserting closers right after orphaned AIMessages
    result = []
    for m in msgs:
        result.append(m)

        if isinstance(m, AIMessage):
            tcs = getattr(m, "tool_calls", None) or []
            orphaned = [tc for tc in tcs if tc["id"] not in responded_ids]
            if orphaned:
                for tc in orphaned:
                    result.append(ToolMessage(
                        tool_call_id=tc["id"],
                        name=tc.get("name", "unknown"),
                        content="[auto-closed: no response was recorded]",
                    ))
                    responded_ids.add(tc["id"])  # mark as handled

    return result


def executor_node(state: AgentState) -> dict:
    tools_for_turn = _read_toolset_for_state(state)
    llm_bound = llm.bind_tools(tools_for_turn + SIGNALING_TOOLS)

    role = state.get("user_role", "customer")
    name = state.get("user_display_name", "User")
    cust = state.get("customer_id", None)

    ctx = {
        "intent": state.get("intent"),
        "order_id": state.get("order_id"),
        "sku": state.get("sku"),
        "size": state.get("size"),
        "qty": state.get("qty"),
        "from_store": state.get("from_store"),
        "to_store": state.get("to_store"),
        "new_status": state.get("new_status"),
        "store_id": state.get("store_id"),
        "reg_username": state.get("reg_username"),
        "reg_pin_code": state.get("reg_pin_code"),
        "reg_display_name": state.get("reg_display_name"),
        "reg_email": state.get("reg_email"),
        # NEW: target customer for manager-on-behalf ordering
        "target_customer_id": state.get("target_customer_id"),
        "target_customer_name": state.get("target_customer_name"),
        "pending_order": state.get("pending_order"),
        "missing_fields": state.get("missing_fields", []),
        "needs_manager_approval": state.get("needs_manager_approval", False),
        "manager_feedback": state.get("manager_feedback"),
        "evidence": state.get("evidence") or {},
    }
    sys = SystemMessage(content=EXECUTOR_PROMPT + f"\n\nSession:\n- user_role={role}\n- user_display_name={name}\n- customer_id={cust}\n\nStateContext:\n{json.dumps(ctx, ensure_ascii=False)}\n")

    raw_msgs = [m for m in state["messages"] if not isinstance(m, SystemMessage)]
    clean_msgs = _sanitize_messages(raw_msgs)
    msgs = [sys] + clean_msgs

    resp = llm_bound.invoke(msgs)
    return {"messages": [resp]}

In [ ]:
def plaintext_guardrail_node(state: AgentState) -> dict:
    # Close any orphaned tool calls before adding the error message
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", None) or []
    closes = [
        ToolMessage(tool_call_id=tc["id"], name=tc["name"], content="SYSTEM: Closed — plaintext guardrail triggered.")
        for tc in tool_calls
    ]
    closes.append(HumanMessage(content="SYSTEM: Invalid output. You MUST call a tool (READ tools or SubmitAnswer/SubmitDraftAction). No plain text."))
    return {"messages": closes}

In [ ]:
def reject_mixed_toolcall_node(state: AgentState) -> dict:
    # Close any orphaned tool calls before adding the error message
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", None) or []
    closes = [
        ToolMessage(tool_call_id=tc["id"], name=tc["name"], content="SYSTEM: Closed — mixed tool call rejected.")
        for tc in tool_calls
    ]
    closes.append(HumanMessage(content="SYSTEM: Do NOT mix SubmitAnswer/SubmitDraftAction with READ tools in the same message. First call READ tools, then after tool results call the final tool."))
    return {"messages": closes}

In [ ]:
def executor_router(state: AgentState) -> Literal["tools_read", "critic", "finalize", "plaintext_guardrail", "reject_mixed"]:
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", None) or []
    if not tool_calls:
        return "plaintext_guardrail"

    names = [tc["name"] for tc in tool_calls]

    allowed_names = {t.name for t in _read_toolset_for_state(state)}
    has_read = any(n in allowed_names for n in names)

    has_submit_answer = "SubmitAnswer" in names
    has_submit_draft = "SubmitDraftAction" in names

    if has_submit_answer and has_submit_draft:
        return "reject_mixed"
    if has_read and (has_submit_answer or has_submit_draft):
        return "reject_mixed"

    if has_read:
        return "tools_read"
    if has_submit_draft:
        return "critic"
    if has_submit_answer:
        return "finalize"

    return "plaintext_guardrail"

In [ ]:
def _slice_last_turn(msgs: List[BaseMessage]) -> List[BaseMessage]:
    last_h = None
    for i in range(len(msgs) - 1, -1, -1):
        if isinstance(msgs[i], HumanMessage):
            last_h = i
            break
    return msgs[last_h:] if last_h is not None else msgs

In [ ]:
def evidence_collector_node(state: AgentState) -> dict:
    ev = dict(state.get("evidence") or {})
    pending = state.get("pending_order")

    turn = _slice_last_turn(state["messages"])
    for m in turn:
        if isinstance(m, ToolMessage) and getattr(m, "name", None):
            # Detect successful order summary → store as pending_order
            if m.name == "prepare_order_summary":
                try:
                    data = json.loads(m.content) if isinstance(m.content, str) else m.content
                    if isinstance(data, dict) and data.get("success") and data.get("all_available"):
                        pending = data
                except (json.JSONDecodeError, TypeError):
                    pass

            # Detect successful order execution → clear pending_order
            elif m.name == "execute_customer_order":
                try:
                    data = json.loads(m.content) if isinstance(m.content, str) else m.content
                    if isinstance(data, dict) and data.get("success"):
                        pending = None
                except (json.JSONDecodeError, TypeError):
                    pass

            # Normal evidence collection
            elif m.name not in ("SubmitAnswer", "SubmitDraftAction"):
                ev[m.name] = (m.content or "")[:500]

    return {"evidence": ev, "pending_order": pending}

In [ ]:
# Block 4 — critic_node (DELETE is now blocked in safety check)

_FORBIDDEN_SQL = {"DROP", "TRUNCATE", "ALTER", "CREATE TABLE", "GRANT", "REVOKE", "DELETE"}

def _sql_is_safe(sql: str) -> tuple:
    """Returns (is_safe: bool, reason: str). Blocks dangerous SQL."""
    upper = sql.upper().strip()
    for keyword in _FORBIDDEN_SQL:
        if keyword in upper:
            return False, f"Forbidden SQL keyword detected: {keyword}"
    if not any(upper.startswith(k) for k in ("SELECT", "INSERT", "UPDATE")):
        return False, f"SQL must start with SELECT, INSERT, or UPDATE. Got: {upper[:30]}..."
    return True, ""


def critic_node(state: AgentState) -> Command[Literal["hitl", "executor", "formatter"]]:
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", None) or []
    draft_call = next((tc for tc in tool_calls if tc["name"] == "SubmitDraftAction"), None)

    if not draft_call:
        closes = _close_last_submit_draft_if_present(state)
        return Command(
            update={
                "messages": closes,
                "pending_user_reply": {
                    "success": False,
                    "action_type": "UNKNOWN",
                    "error_code": "MISSING_DRAFT_ACTION",
                    "raw_error": "No SubmitDraftAction tool call found.",
                    "retryable": True,
                    "source": "DB",
                    "fallback_text": "❌ I couldn't prepare that action request. Please try again."
                },
            },
            goto="formatter",
        )

    draft = draft_call.get("args", {}) or {}
    action_type = (draft.get("action_type") or "").upper().strip()

    def _repair_loop(reason: str, missing_fields: List[str]) -> Command[Literal["executor"]]:
        close = ToolMessage(
            tool_call_id=draft_call["id"],
            name="SubmitDraftAction",
            content=f"CRITIC REJECTED: {reason}",
        )
        return Command(
            update={
                "messages": [close],
                "missing_fields": missing_fields,
                "draft_action": None,
                "plan": None,
            },
            goto="executor",
        )

    def _final_reject(error_code: str, raw_error: str, retryable: bool, extra: Optional[dict] = None) -> Command[Literal["formatter"]]:
        close = ToolMessage(
            tool_call_id=draft_call["id"],
            name="SubmitDraftAction",
            content=f"CRITIC REJECTED: {raw_error}",
        )
        payload = {
            "success": False,
            "action_type": action_type or "UNKNOWN",
            "error_code": error_code,
            "raw_error": raw_error,
            "retryable": retryable,
            "source": "DB",
        }
        if extra:
            payload.update(extra)

        return Command(
            update={
                "messages": [close],
                "draft_action": None,
                "plan": None,
                "missing_fields": [],
                "pending_user_reply": payload,
                "needs_manager_approval": False,
                "manager_decision": None,
                "manager_feedback": None,
            },
            goto="formatter",
        )

    # --- ROLE CHECK ---
    if state.get("user_role") != "manager":
        return _final_reject(
            error_code="MANAGER_ONLY",
            raw_error="This is a manager-only operation.",
            retryable=False,
            extra={"draft": draft},
        )

    # =================== TRANSFER ===================
    if action_type == "TRANSFER":
        missing = [k for k in ["from_store", "to_store", "sku", "qty"] if draft.get(k) in (None, "", [])]
        if missing:
            return _repair_loop(
                reason="; ".join([f"missing field: {k}" for k in missing]),
                missing_fields=missing,
            )

        try:
            qty = int(draft.get("qty"))
        except Exception:
            return _final_reject(
                error_code="INVALID_QTY_TYPE",
                raw_error="Quantity must be an integer.",
                retryable=True,
                extra={"draft": draft},
            )

        if qty < 1:
            return _final_reject(
                error_code="INVALID_QTY_MIN",
                raw_error="Quantity must be at least 1.",
                retryable=True,
                extra={"draft": draft, "qty": qty},
            )

        if qty > 20:
            return _final_reject(
                error_code="TRANSFER_LIMIT_EXCEEDED",
                raw_error="Quantity exceeds the maximum transfer limit of 20 units.",
                retryable=False,
                extra={"draft": draft, "qty": qty, "max_qty": 20},
            )

        from_store = draft["from_store"]
        to_store = draft["to_store"]
        sku = draft["sku"]

        try:
            conn = get_db_conn()
            cur = conn.cursor()
            cur.execute(
                "SELECT on_hand, reserved FROM inventory WHERE store_id=%s AND sku=%s",
                (from_store, sku),
            )
            row = cur.fetchone()

            if not row:
                return _final_reject(
                    error_code="SKU_NOT_FOUND_IN_SOURCE",
                    raw_error=f"No inventory record for {sku} in {from_store}.",
                    retryable=False,
                    extra={"draft": draft, "sku": sku, "from_store": from_store, "to_store": to_store},
                )

            on_hand, reserved = row[0] or 0, row[1] or 0
            available = int(on_hand) - int(reserved)

            if available < qty:
                return _final_reject(
                    error_code="INSUFFICIENT_STOCK",
                    raw_error=f"Insufficient available stock. Available={available}, Requested={qty}.",
                    retryable=False,
                    extra={
                        "draft": draft, "sku": sku, "qty": qty,
                        "available": available, "from_store": from_store, "to_store": to_store,
                    },
                )
        except Exception as e:
            return _final_reject(
                error_code="DB_VERIFICATION_FAILED",
                raw_error=f"DB verification failed: {type(e).__name__}: {e}",
                retryable=True,
                extra={"draft": draft},
            )
        finally:
            if "conn" in locals() and conn:
                conn.close()

        return Command(
            update={
                "draft_action": draft,
                "plan": draft.get("rationale", ""),
                "missing_fields": [],
            },
            goto="hitl",
        )

    # =================== STATUS_UPDATE ===================
    if action_type == "STATUS_UPDATE":
        missing = [k for k in ["order_id", "new_status"] if draft.get(k) in (None, "", [])]
        if missing:
            return _repair_loop(
                reason="; ".join([f"missing field: {k}" for k in missing]),
                missing_fields=missing,
            )

        order_id = draft["order_id"]
        try:
            conn = get_db_conn()
            cur = conn.cursor()
            cur.execute("SELECT 1 FROM orders WHERE order_id=%s", (order_id,))
            if not cur.fetchone():
                return _final_reject(
                    error_code="ORDER_NOT_FOUND",
                    raw_error=f"Order {order_id} was not found.",
                    retryable=False,
                    extra={"draft": draft, "order_id": order_id},
                )
        except Exception as e:
            return _final_reject(
                error_code="DB_VERIFICATION_FAILED",
                raw_error=f"DB verification failed: {type(e).__name__}: {e}",
                retryable=True,
                extra={"draft": draft, "order_id": order_id},
            )
        finally:
            if "conn" in locals() and conn:
                conn.close()

        return Command(
            update={
                "draft_action": draft,
                "plan": draft.get("rationale", ""),
                "missing_fields": [],
            },
            goto="hitl",
        )

    # =================== DB_WRITE ===================
    if action_type == "DB_WRITE":
        sql_query = (draft.get("sql_query") or "").strip()
        description = (draft.get("description") or "").strip()

        if not sql_query:
            return _repair_loop(
                reason="sql_query is missing or empty. You must call sql_writer first, then include the result.",
                missing_fields=["sql_query"],
            )

        if not description:
            return _repair_loop(
                reason="description is missing. Provide a human-readable summary of what the SQL does.",
                missing_fields=["description"],
            )

        sql_clean = _strip_sql_fences(sql_query)

        is_safe, safety_reason = _sql_is_safe(sql_clean)
        if not is_safe:
            return _final_reject(
                error_code="UNSAFE_SQL",
                raw_error=f"SQL safety check failed: {safety_reason}",
                retryable=False,
                extra={"draft": draft, "sql_query": sql_clean},
            )

        draft["sql_query"] = sql_clean

        return Command(
            update={
                "draft_action": draft,
                "plan": draft.get("rationale", ""),
                "missing_fields": [],
            },
            goto="hitl",
        )

    # =================== UNKNOWN ===================
    return _final_reject(
        error_code="UNKNOWN_ACTION_TYPE",
        raw_error=f"Unknown action_type: {action_type}",
        retryable=False,
        extra={"draft": draft},
    )

In [ ]:
# Block 3 — hitl_node: always interrupt for manager write ops (no toggle)
def hitl_node(state: AgentState) -> Command[Literal["apply_write", "executor"]]:
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", None) or []
    draft_call = next((tc for tc in tool_calls if tc["name"] == "SubmitDraftAction"), None)

    if not draft_call:
        return Command(update={"messages": []}, goto="executor")

    if state.get("user_role") != "manager":
        close = ToolMessage(tool_call_id=draft_call["id"], name="SubmitDraftAction", content="REJECTED: manager-only operation.")
        return Command(update={"messages": [close], "needs_manager_approval": False}, goto="executor")

    # IMPORTANT: use the current call args ONLY
    draft = (draft_call.get("args", {}) or {})

    # Always interrupt for write operations (HITL always on)
    fb = interrupt({
        "draft_action": draft,
        "risk_level": draft.get("risk_level"),
        "rationale": draft.get("rationale"),
        "evidence": state.get("evidence") or {},
        "tool_call_id": draft_call["id"],
    })
    decision = (fb.get("decision") or fb.get("feedback") or "").strip()
    feedback = (fb.get("feedback") or "").strip()

    approved = decision.upper() in ("APPROVE", "APPROVED", "YES", "OK")

    if not approved:
        close = ToolMessage(tool_call_id=draft_call["id"], name="SubmitDraftAction", content=f"REJECTED: {feedback or decision}".strip())
        return Command(
            update={
                "messages": [close],
                "needs_manager_approval": True,
                "manager_decision": "reject",
                "manager_feedback": (feedback or decision).strip(),
            },
            goto="executor",
        )

    close = ToolMessage(tool_call_id=draft_call["id"], name="SubmitDraftAction", content="APPROVED")

    return Command(
        update={
            "messages": [close],
            "needs_manager_approval": False,
            "manager_decision": "approve",
            "manager_feedback": feedback,
            "draft_action": draft,
        },
        goto="apply_write",
    )

In [ ]:
# Block — REPLACE apply_write_node
def apply_write_node(state: AgentState) -> Command[Literal["formatter"]]:
    draft = dict(state.get("draft_action") or {})
    approved_by = state.get("user_display_name") or "manager"
    action_type = (draft.get("action_type") or "").upper().strip()

    payload = {
        "success": False,
        "action_type": action_type or "UNKNOWN",
        "source": "DB",
        "retryable": False,
        "draft": draft,
    }

    try:
        if not action_type:
            payload.update({
                "error_code": "MISSING_ACTION_TYPE",
                "raw_error": "Approved draft_action is missing action_type.",
                "retryable": True,
                "fallback_text": "❌ I couldn't complete that action because the approved request data was incomplete."
            })

        elif action_type == "TRANSFER":
            from_store = draft.get("from_store")
            to_store = draft.get("to_store")
            sku = draft.get("sku")

            try:
                qty = int(draft.get("qty"))
            except Exception:
                qty = None

            missing = [
                k for k, v in [
                    ("from_store", from_store),
                    ("to_store", to_store),
                    ("sku", sku),
                    ("qty", qty),
                ]
                if v in (None, "", [])
            ]

            if missing:
                payload.update({
                    "error_code": "INVALID_TRANSFER_DRAFT",
                    "raw_error": f"Invalid transfer draft: missing/invalid {', '.join(missing)}.",
                    "retryable": True,
                })
            else:
                res = execute_transfer.invoke({
                    "from_store": from_store,
                    "to_store": to_store,
                    "sku": sku,
                    "qty": qty,
                    "approved_by": approved_by,
                })

                if res.get("success"):
                    payload.update({
                        "success": True,
                        "error_code": None,
                        "raw_error": None,
                        "qty": qty,
                        "sku": sku,
                        "from_store": from_store,
                        "to_store": to_store,
                        "from_available_before": res.get("from_available_before"),
                        "from_available_after": res.get("from_available_after"),
                        "to_available_after": res.get("to_available_after"),
                        "fallback_text": f"✅ Transfer completed successfully: moved {qty} units of {sku} from {from_store} to {to_store}."
                    })
                else:
                    payload.update({
                        "error_code": "TRANSFER_EXECUTION_FAILED",
                        "raw_error": str(res.get("error") or "Transfer execution failed."),
                        "qty": qty,
                        "sku": sku,
                        "from_store": from_store,
                        "to_store": to_store,
                        "retryable": False,
                    })

        elif action_type == "STATUS_UPDATE":
            order_id = draft.get("order_id")
            new_status = draft.get("new_status")

            missing = [
                k for k, v in [
                    ("order_id", order_id),
                    ("new_status", new_status),
                ]
                if v in (None, "", [])
            ]

            if missing:
                payload.update({
                    "error_code": "INVALID_STATUS_DRAFT",
                    "raw_error": f"Invalid status update draft: missing {', '.join(missing)}.",
                    "retryable": True,
                })
            else:
                res = update_order_status.invoke({
                    "order_id": order_id,
                    "new_status": new_status,
                    "approved_by": approved_by,
                })

                if res.get("success"):
                    payload.update({
                        "success": True,
                        "error_code": None,
                        "raw_error": None,
                        "order_id": res.get("order_id", order_id),
                        "old_status": res.get("old_status"),
                        "new_status": res.get("new_status", new_status),
                        "store_id": res.get("store_id"),
                        "warning_text": res.get("warning_text"),
                        "fallback_text": (
                            f"✅ Order {res.get('order_id', order_id)} was updated successfully: "
                            f"status changed from {res.get('old_status')} to {res.get('new_status', new_status)}."
                        ),
                    })
                else:
                    payload.update({
                        "error_code": "STATUS_UPDATE_EXECUTION_FAILED",
                        "raw_error": str(res.get("error") or "Status update failed."),
                        "order_id": order_id,
                        "new_status": new_status,
                        "retryable": False,
                    })

        # =================== DB_WRITE (FIXED: deterministic auto-inventory) ===================
        elif action_type == "DB_WRITE":
            sql_query = (draft.get("sql_query") or "").strip()
            description = (draft.get("description") or "").strip()
            sku = (draft.get("sku") or "").strip()

            # DETERMINISTIC: detect product creation from the SQL itself
            # Don't rely on LLM setting auto_create_inventory flag
            sql_upper = sql_query.upper()
            is_product_insert = ("INSERT" in sql_upper and "PRODUCTS" in sql_upper)
            auto_inv = bool(draft.get("auto_create_inventory", False)) or is_product_insert

            # If SKU not in draft, try to extract it from the SQL
            if auto_inv and not sku:
                import re
                # Match SKU patterns like 'TST-RED-43' in VALUES clause
                sku_match = re.search(r"VALUES\s*\(\s*'([A-Z0-9]+-[A-Z0-9]+-[A-Z0-9]+-?\d*)'", sql_query, re.IGNORECASE)
                if sku_match:
                    sku = sku_match.group(1)

            if not sql_query:
                payload.update({
                    "error_code": "MISSING_SQL",
                    "raw_error": "No SQL query in the approved draft.",
                    "retryable": True,
                })
            else:
                conn = get_db_conn()
                try:
                    cur = conn.cursor()

                    # Execute the main SQL
                    cur.execute(sql_query)
                    main_rowcount = cur.rowcount

                    # Auto-create inventory rows for all stores if this is a product creation
                    inventory_stores_created = []
                    if auto_inv and sku:
                        # Query from STORES table (not inventory) to get ALL stores
                        cur.execute("SELECT store_id FROM stores ORDER BY store_id")
                        all_stores = [row[0] for row in cur.fetchall()]

                        for sid in all_stores:
                            try:
                                cur.execute(
                                    """
                                    INSERT INTO inventory (store_id, sku, on_hand, reserved)
                                    VALUES (%s, %s, 0, 0)
                                    ON CONFLICT (store_id, sku) DO NOTHING
                                    """,
                                    (sid, sku),
                                )
                                if cur.rowcount > 0:
                                    inventory_stores_created.append(sid)
                            except Exception:
                                pass

                    conn.commit()

                    fallback = f"✅ SQL executed successfully. Rows affected: {main_rowcount}."
                    if description:
                        fallback = f"✅ {description} — completed. Rows affected: {main_rowcount}."
                    if inventory_stores_created:
                        fallback += (
                            f" Inventory initialized (on_hand=0) at {len(inventory_stores_created)} "
                            f"store(s): {', '.join(inventory_stores_created)}."
                        )

                    payload.update({
                        "success": True,
                        "error_code": None,
                        "raw_error": None,
                        "sql_query": sql_query,
                        "description": description,
                        "rows_affected": main_rowcount,
                        "inventory_stores_created": inventory_stores_created,
                        "fallback_text": fallback,
                    })

                except Exception as e:
                    conn.rollback()
                    payload.update({
                        "error_code": "DB_WRITE_EXECUTION_FAILED",
                        "raw_error": f"{type(e).__name__}: {e}",
                        "sql_query": sql_query,
                        "retryable": True,
                    })
                finally:
                    conn.close()

        else:
            payload.update({
                "error_code": "UNKNOWN_ACTION_TYPE",
                "raw_error": f"Unknown action_type: {action_type}",
                "retryable": False,
            })

    except Exception as e:
        payload.update({
            "error_code": "INTERNAL_EXECUTION_ERROR",
            "raw_error": f"{type(e).__name__}: {e}",
            "retryable": True,
        })

    return Command(
        update={
            "pending_user_reply": payload,

            "intent": None,
            "sku": None,
            "size": None,
            "qty": None,
            "from_store": None,
            "to_store": None,
            "new_status": None,
            "order_id": None,
            "store_id": None,

            "draft_action": None,
            "evidence": {},
            "manager_decision": None,
            "manager_feedback": None,
            "plan": None,
            "needs_manager_approval": False,
        },
        goto="formatter",
    )

In [ ]:
# Block 6I — Finalize (close tool calls) + Build Graph
def finalize_node(state: AgentState) -> dict:
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", None) or []
    closes = [ToolMessage(tool_call_id=tc["id"], name=tc["name"], content="Delivered.") for tc in tool_calls]
    return {"messages": closes}

In [ ]:
# Block — REPLACE graph build

builder = StateGraph(AgentState)

# nodes
builder.add_node("planner", planner_node)
builder.add_node("planner_apply", planner_apply_node)
builder.add_node("ask_user", ask_user_node)

builder.add_node("executor", executor_node)
builder.add_node("tools_read", ToolNode(READ_TOOL_NODE_ALL))
builder.add_node("evidence", evidence_collector_node)

builder.add_node("plaintext_guardrail", plaintext_guardrail_node)
builder.add_node("reject_mixed", reject_mixed_toolcall_node)

builder.add_node("critic", critic_node)
builder.add_node("hitl", hitl_node)
builder.add_node("apply_write", apply_write_node)
builder.add_node("formatter", formatter_node)

builder.add_node("finalize", finalize_node)

# edges
builder.add_edge(START, "planner")
builder.add_edge("planner", "planner_apply")

builder.add_conditional_edges(
    "planner_apply",
    route_after_planner,
    {
        "ask_user": "ask_user",
        "executor": "executor",
    },
)

builder.add_edge("ask_user", "finalize")

builder.add_conditional_edges(
    "executor",
    executor_router,
    {
        "tools_read": "tools_read",
        "critic": "critic",
        "finalize": "finalize",
        "plaintext_guardrail": "plaintext_guardrail",
        "reject_mixed": "reject_mixed",
    },
)

builder.add_edge("tools_read", "evidence")
builder.add_edge("evidence", "executor")

builder.add_edge("plaintext_guardrail", "executor")
builder.add_edge("reject_mixed", "executor")

# write flow
# critic_node uses Command(goto=...) -> hitl / executor / formatter
# hitl_node uses Command(goto=...) -> apply_write (approve) / executor (reject)
# NO static edges for hitl or apply_write — Command handles routing
builder.add_edge("apply_write", "formatter")
builder.add_edge("formatter", "finalize")

builder.add_edge("finalize", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

print("✅ Graph compiled (Planner → Executor → Critic → HITL/Formatter → Finalize)")

✅ Graph compiled (Planner → Executor → Critic → HITL/Formatter → Finalize)


In [ ]:
# Block 7 — Gradio UI (Login + Chat + HITL admin panel)
import gradio as gr

def _extract_latest_submit_answer(msgs: List[BaseMessage]) -> Optional[dict]:
    # Find most recent AI tool call to SubmitAnswer
    for m in reversed(msgs):
        if isinstance(m, AIMessage):
            tcs = getattr(m, "tool_calls", None) or []
            for tc in tcs:
                if tc.get("name") == "SubmitAnswer":
                    return tc.get("args", {})
    return None

In [ ]:
def _tool_log_from_turn(turn_msgs: List[BaseMessage]) -> str:
    used = []
    for m in turn_msgs:
        if isinstance(m, ToolMessage) and getattr(m, "name", None):
            if m.name not in ("SubmitAnswer", "SubmitDraftAction"):
                used.append(m.name)
    used = list(dict.fromkeys(used))
    return "Tools used: " + (", ".join(used) if used else "None")

In [ ]:
def _slice_last_turn(msgs: List[BaseMessage]) -> List[BaseMessage]:
    # Slice messages from the last HumanMessage to the end
    last_h = None
    for i in range(len(msgs) - 1, -1, -1):
        if isinstance(msgs[i], HumanMessage):
            last_h = i
            break
    return msgs[last_h:] if last_h is not None else msgs

In [ ]:
def new_chat():
    return str(uuid4()), [], gr.update(visible=False), "", "", "Tools used: None"

In [ ]:
# Block A — helper: extract latest friendly answer
def _extract_latest_submit_answer(msgs: List[BaseMessage]) -> Optional[dict]:
    for m in reversed(msgs):
        if isinstance(m, AIMessage):
            tcs = getattr(m, "tool_calls", None) or []
            for tc in tcs:
                if tc.get("name") == "SubmitAnswer":
                    return tc.get("args", {})
    return None

In [ ]:
# Block B — helper: extract critic rejection / HITL rejection as user-friendly text
def _extract_latest_backend_reason(msgs: List[BaseMessage]) -> Optional[str]:
    for m in reversed(msgs):
        if isinstance(m, ToolMessage) and getattr(m, "name", None) == "SubmitDraftAction":
            content = (m.content or "").strip()

            if content.startswith("CRITIC REJECTED:"):
                reason = content.replace("CRITIC REJECTED:", "", 1).strip()

                if "qty exceeds max transfer limit (20)" in reason:
                    return "❌ Transfer failed: quantity exceeds the maximum transfer limit of 20 units."

                if "insufficient available stock" in reason:
                    return "❌ Transfer failed: not enough available stock in the source store."

                if "no inventory record" in reason:
                    return "❌ Transfer failed: the SKU does not exist in the source store inventory."

                return f"❌ Transfer failed: {reason}"

            if content.startswith("REJECTED:"):
                reason = content.replace("REJECTED:", "", 1).strip()
                return f"❌ Action was rejected: {reason}"

    return None

In [ ]:
import re

def _extract_registration_result(turn_msgs: List[BaseMessage]) -> Optional[dict]:
    for m in turn_msgs:
        if isinstance(m, ToolMessage) and getattr(m, "name", None) == "register_new_user":
            try:
                data = json.loads(m.content) if isinstance(m.content, str) else m.content
                if isinstance(data, dict) and data.get("success"):
                    return data
            except (json.JSONDecodeError, TypeError):
                pass
    return None

In [ ]:
# Block 9 — ADD this helper near your other UI helper functions, before chat_handler
def _extract_email_warning(turn_msgs: List[BaseMessage]) -> Optional[str]:
    for m in reversed(turn_msgs):
        if isinstance(m, ToolMessage) and getattr(m, "name", None) == "execute_customer_order":
            try:
                data = json.loads(m.content) if isinstance(m.content, str) else m.content
            except (json.JSONDecodeError, TypeError):
                data = None

            if isinstance(data, dict):
                warning = (data.get("email_warning") or "").strip()
                if warning:
                    return f"⚠️ {warning}"
    return None

In [ ]:
# Block — REPLACE chat_handler
def chat_handler(user_text, history, thread_id, session_state):
    history = history or []
    session_state = session_state or {}

    if not user_text:
        return (
            "",
            history,
            thread_id,
            gr.update(visible=False),
            "",
            "",
            "Tools used: None",
            session_state,
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
        )

    user_role = session_state.get("user_role", "customer")
    display_name = session_state.get("display_name", "Guest")
    customer_id = session_state.get("customer_id")

    cfg = {"configurable": {"thread_id": thread_id}}

    # 1) If a review is already pending, block new input
    try:
        pending_snapshot = graph.get_state(cfg)
        if pending_snapshot.tasks and pending_snapshot.tasks[0].interrupts:
            int_val = pending_snapshot.tasks[0].interrupts[0].value
            draft = int_val.get("draft_action", {})
            msgs = (pending_snapshot.values or {}).get("messages", []) or []

            history = history + [
                {"role": "user", "content": user_text},
                {"role": "assistant", "content": "⏳ A manager review is still pending. Please approve or reject it first."},
            ]

            return (
                "",
                history,
                thread_id,
                gr.update(visible=True),
                json.dumps(draft, indent=2, ensure_ascii=False),
                "",
                _tool_log_from_turn(_slice_last_turn(msgs)),
                session_state,
                gr.update(),
                gr.update(),
                gr.update(),
                gr.update(),
                gr.update(),
            )
    except Exception:
        pass

    # 2) Preserve lock only for write flows
    lock = False
    try:
        snap0 = graph.get_state(cfg)
        prev = (snap0.values or {})
        prev_intent = prev.get("intent")
        lock = bool(prev.get("needs_manager_approval", False)) and (
            prev_intent in ("transfer", "status_update")
        )
    except Exception:
        lock = False

    history = history + [{"role": "user", "content": user_text}]

    init = {
        "messages": [HumanMessage(content=user_text)],
        "user_role": user_role,
        "user_display_name": display_name,
        "customer_id": customer_id,
        "needs_manager_approval": lock,
    }

    result = None
    invoke_error = None
    try:
        result = graph.invoke(init, config=cfg)
    except Exception as e:
        invoke_error = e
        print(f"🔴 INVOKE ERROR: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()

    try:
        snap = graph.get_state(cfg)
    except Exception:
        snap = None

    if snap and snap.tasks and snap.tasks[0].interrupts:
        int_val = snap.tasks[0].interrupts[0].value
        draft = int_val.get("draft_action", {})
        msgs = (snap.values or {}).get("messages", []) or []

        return (
            "",
            history,
            thread_id,
            gr.update(visible=True),
            json.dumps(draft, indent=2, ensure_ascii=False),
            "",
            _tool_log_from_turn(_slice_last_turn(msgs)),
            session_state,
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
        )

    msgs = []
    if snap and getattr(snap, "values", None):
        msgs = snap.values.get("messages", []) or []
    elif result:
        msgs = result.get("messages", []) or []

    turn_msgs = _slice_last_turn(msgs)

    ans = _extract_latest_submit_answer(turn_msgs)
    if ans:
        answer_text = ans.get("answer", "")
        email_warning = _extract_email_warning(turn_msgs)
        if email_warning:
            answer_text = f"{answer_text}\n\n{email_warning}"

        # --- REGISTRATION DETECTION ---
        reg_data = _extract_registration_result(turn_msgs)
        if reg_data:
            session_state["logged_in"] = True
            session_state["user_role"] = "customer"
            session_state["display_name"] = reg_data.get("display_name", "Customer")
            session_state["customer_id"] = reg_data.get("customer_id", "")

            history = history + [{"role": "assistant", "content": answer_text}]
            tools_used = _tool_log_from_turn(turn_msgs)

            return (
                "",
                history,
                thread_id,
                gr.update(visible=False),
                "",
                "",
                tools_used,
                session_state,
                gr.update(value=reg_data.get("display_name", "Customer")),
                gr.update(value="customer"),
                gr.update(value=reg_data.get("customer_id", ""), visible=True),
                gr.update(visible=False),
                gr.update(visible=True),
            )

        # --- NORMAL ANSWER ---
        history = history + [{"role": "assistant", "content": answer_text}]
        tools_used = _tool_log_from_turn(turn_msgs)

        return (
            "",
            history,
            thread_id,
            gr.update(visible=False),
            "",
            "",
            tools_used,
            session_state,
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
        )

    backend_reason = _extract_latest_backend_reason(turn_msgs)
    if backend_reason:
        history = history + [{"role": "assistant", "content": backend_reason}]
        tools_used = _tool_log_from_turn(turn_msgs)

        return (
            "",
            history,
            thread_id,
            gr.update(visible=False),
            "",
            "",
            tools_used,
            session_state,
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
        )

    if invoke_error:
        history = history + [
            {
                "role": "assistant",
                "content": "❌ The action could not be completed because of an internal tool-flow error. Please try again.",
            }
        ]
        return (
            "",
            history,
            str(uuid4()),
            gr.update(visible=False),
            "",
            "",
            "Tools used: None (invoke failed)",
            session_state,
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
        )

    history = history + [{"role": "assistant", "content": "❌ No answer was produced."}]
    return (
        "",
        history,
        thread_id,
        gr.update(visible=False),
        "",
        "",
        "Tools used: None",
        session_state,
        gr.update(),
        gr.update(),
        gr.update(),
        gr.update(),
        gr.update(),
    )

In [ ]:
# Block — REPLACE admin_decide_handler
def admin_decide_handler(decision, admin_feedback, history, thread_id):
    history = history or []
    cfg = {"configurable": {"thread_id": thread_id}}
    resume_payload = {"decision": decision, "feedback": admin_feedback or ""}

    result = None
    resume_error = None

    try:
        result = graph.invoke(Command(resume=resume_payload), config=cfg)
    except Exception as e:
        resume_error = e

    try:
        snap = graph.get_state(cfg)
    except Exception:
        snap = None

    # 1) If still paused, keep admin panel open with the CURRENT draft
    if snap and snap.tasks and snap.tasks[0].interrupts:
        int_val = snap.tasks[0].interrupts[0].value
        draft = int_val.get("draft_action", {})
        msgs = (snap.values or {}).get("messages", []) or []

        return (
            history,
            thread_id,
            gr.update(visible=True),
            json.dumps(draft, indent=2, ensure_ascii=False),
            admin_feedback,
            _tool_log_from_turn(_slice_last_turn(msgs)),
        )

    # 2) Prefer parsing checkpoint/result messages
    msgs = []
    if snap and getattr(snap, "values", None):
        msgs = snap.values.get("messages", []) or []
    elif result:
        msgs = result.get("messages", []) or []

    # Final normal answer
    ans = _extract_latest_submit_answer(msgs)
    if ans:
        history = history + [{"role": "assistant", "content": ans.get("answer", "")}]
        tools_used = _tool_log_from_turn(_slice_last_turn(msgs))
        return history, thread_id, gr.update(visible=False), "", "", tools_used

    # Final rejection / critic message
    backend_reason = _extract_latest_backend_reason(msgs)
    if backend_reason:
        history = history + [{"role": "assistant", "content": backend_reason}]
        tools_used = _tool_log_from_turn(_slice_last_turn(msgs))
        return history, thread_id, gr.update(visible=False), "", "", tools_used

    # Resume failed and produced nothing useful -> reset thread so it doesn't stay jammed
    if resume_error:
        history = history + [{
            "role": "assistant",
            "content": "❌ The review action failed because of an internal tool-flow error. I reset the chat thread so it won’t stay stuck. Please try again."
        }]
        return history, str(uuid4()), gr.update(visible=False), "", "", "Tools used: None (resume failed)"

    history = history + [{"role": "assistant", "content": "❌ No answer was produced after review."}]
    return history, thread_id, gr.update(visible=False), "", "", "Tools used: None"

In [ ]:
# Block 5 — Gradio UI (Login-first, then Chat)
def login_handler(username, pin, session_state, thread_id, history_state):
    session_state = session_state or {}

    if not username or not pin:
        return (
            "Enter username + PIN",
            session_state,
            thread_id,
            history_state,
            gr.update(visible=True),
            gr.update(visible=False),
            "Guest",
            "customer",
            gr.Textbox(value="", visible=True),
        )

    res = authenticate_user.invoke({"username": username, "pin_code": pin})
    if not res.get("success"):
        return (
            f"Login failed: {res.get('error')}",
            session_state,
            thread_id,
            history_state,
            gr.update(visible=True),
            gr.update(visible=False),
            "Guest",
            "customer",
            gr.Textbox(value="", visible=True),
        )

    user = res["user"]
    session_state["logged_in"] = True
    session_state["user_role"] = user["role"]
    session_state["display_name"] = user["display_name"]
    session_state["customer_id"] = user.get("customer_id")

    new_thread = str(uuid4())
    new_history = []

    is_manager = user["role"] == "manager"

    return (
        f"✅ Logged in as {user['display_name']} ({user['role']})",
        session_state,
        new_thread,
        new_history,
        gr.update(visible=False),
        gr.update(visible=True),
        user["display_name"],
        user["role"],
        gr.Textbox(value=user.get("customer_id") or "", visible=not is_manager),
    )

In [ ]:
# Patch 1 — logout_handler (clears everything, shows customer_id again)
def logout_handler(session_state):
    session_state = {}
    return (
        "Logged out.",
        session_state,
        str(uuid4()),
        [],
        gr.update(visible=True),    # show login view
        gr.update(visible=False),   # hide chat view
        "Guest",
        "customer",
        gr.update(value="", visible=True),   # customer_id — visible again
        gr.update(value=""),        # clear username textbox
        gr.update(value=""),        # clear pin textbox
    )

In [ ]:
# Block — Gradio UI (Styled: compact login + wide chat)

def guest_handler(session_state, thread_id):
    """Enter chat as guest — no login required."""
    session_state = {
        "logged_in": False,
        "user_role": "guest",
        "display_name": "Guest",
        "customer_id": None,
    }
    new_thread = str(uuid4())
    new_history = []
    return (
        session_state,
        new_thread,
        new_history,
        gr.update(visible=False),
        gr.update(visible=True),
        "Guest",
        "guest",
        gr.update(value="", visible=True),   # customer_id — visible for guests
    )

In [ ]:
CUSTOM_CSS = """
/* =============================================================
   GRADIO CSS VARIABLE OVERRIDES — forces dark theme regardless
   of browser light/dark preference.
   ============================================================= */
:root, :host,
*[data-theme="light"], *[data-theme="dark"],
.gradio-container,
.dark, .light {
    --body-background-fill: #0a0a0f !important;
    --background-fill-primary: #12121a !important;
    --background-fill-secondary: #12121a !important;
    --block-background-fill: #12121a !important;
    --block-border-color: #2a2a3d !important;
    --block-label-background-fill: #12121a !important;
    --block-label-border-color: #2a2a3d !important;
    --block-label-text-color: #f0f0f5 !important;
    --block-title-text-color: #f0f0f5 !important;
    --body-text-color: #f0f0f5 !important;
    --body-text-color-subdued: #8888a0 !important;
    --input-background-fill: #1a1a28 !important;
    --input-border-color: #2a2a3d !important;
    --input-text-color: #f0f0f5 !important;
    --input-placeholder-color: #8f90a6 !important;
    --button-primary-background-fill: linear-gradient(135deg, #f97316, #ea580c) !important;
    --button-primary-text-color: #ffffff !important;
    --button-secondary-background-fill: transparent !important;
    --button-secondary-text-color: #f0f0f5 !important;
    --button-secondary-border-color: #2a2a3d !important;
    --panel-background-fill: #12121a !important;
    --panel-border-color: #2a2a3d !important;
    --color-accent: #f97316 !important;
    --color-accent-soft: rgba(249,115,22,0.15) !important;
    --shadow-drop: none !important;
    --shadow-drop-lg: none !important;
    --chatbot-body-background-fill: #0a0a0f !important;
    --checkbox-background-color: #1a1a28 !important;
    --checkbox-border-color: #2a2a3d !important;
    --checkbox-label-background-fill: #12121a !important;
    --table-even-background-fill: #12121a !important;
    --table-odd-background-fill: #0a0a0f !important;
    --table-border-color: #2a2a3d !important;
    --border-color-primary: #2a2a3d !important;
    --border-color-accent: #f97316 !important;
    --link-text-color: #f97316 !important;
    --link-text-color-hover: #fb923c !important;
    --neutral-50: #f0f0f5 !important;
    --neutral-100: #d0d0dd !important;
    --neutral-200: #8888a0 !important;
    --neutral-300: #6a6a80 !important;
    --neutral-400: #55556a !important;
    --neutral-500: #44445a !important;
    --neutral-600: #2a2a3d !important;
    --neutral-700: #1a1a28 !important;
    --neutral-800: #12121a !important;
    --neutral-900: #0a0a0f !important;
    --neutral-950: #050508 !important;
}

@media (prefers-color-scheme: light) {
    :root, :host,
    *[data-theme="light"],
    .gradio-container,
    .light {
        --body-background-fill: #0a0a0f !important;
        --background-fill-primary: #12121a !important;
        --background-fill-secondary: #12121a !important;
        --block-background-fill: #12121a !important;
        --block-border-color: #2a2a3d !important;
        --block-label-background-fill: #12121a !important;
        --block-label-border-color: #2a2a3d !important;
        --block-label-text-color: #f0f0f5 !important;
        --block-title-text-color: #f0f0f5 !important;
        --body-text-color: #f0f0f5 !important;
        --body-text-color-subdued: #8888a0 !important;
        --input-background-fill: #1a1a28 !important;
        --input-border-color: #2a2a3d !important;
        --input-text-color: #f0f0f5 !important;
        --input-placeholder-color: #8f90a6 !important;
        --panel-background-fill: #12121a !important;
        --panel-border-color: #2a2a3d !important;
        --color-accent: #f97316 !important;
        --shadow-drop: none !important;
        --shadow-drop-lg: none !important;
        --chatbot-body-background-fill: #0a0a0f !important;
        --border-color-primary: #2a2a3d !important;
        --neutral-50: #f0f0f5 !important;
        --neutral-100: #d0d0dd !important;
        --neutral-200: #8888a0 !important;
        --neutral-300: #6a6a80 !important;
        --neutral-400: #55556a !important;
        --neutral-500: #44445a !important;
        --neutral-600: #2a2a3d !important;
        --neutral-700: #1a1a28 !important;
        --neutral-800: #12121a !important;
        --neutral-900: #0a0a0f !important;
        --neutral-950: #050508 !important;
    }
}

/* ===== GLOBAL ===== */
html, body {
    background: #0a0a0f !important;
    color: #f0f0f5 !important;
    margin: 0 !important;
    padding: 0 !important;
}

body {
    overflow-x: hidden !important;
}

.gradio-container {
    background: #0a0a0f !important;
    font-family: 'Segoe UI', system-ui, -apple-system, sans-serif !important;
    min-height: 100vh;
    max-width: 100% !important;
    padding: 0 !important;
    color: #f0f0f5 !important;
}

/* kill default white wrappers / cards / fieldsets */
.gradio-container .contain,
.gradio-container .main,
.gradio-container .wrap,
.gradio-container .block,
.gradio-container .gr-block,
.gradio-container .gr-panel,
.gradio-container .gr-box,
.gradio-container .gr-form,
.gradio-container .gr-group,
.gradio-container .gr-column,
.gradio-container .gr-row,
.gradio-container .form,
.gradio-container fieldset {
    background: transparent !important;
    box-shadow: none !important;
    border-color: transparent !important;
}

.gradio-container .block,
.gradio-container .gr-block,
.gradio-container .gr-form,
.gradio-container .gr-group,
.gradio-container fieldset {
    border: none !important;
    outline: none !important;
}

.gradio-container > .main,
.gradio-container > .main > .wrap,
.gradio-container .contain {
    max-width: 100% !important;
    background: transparent !important;
    padding: 0 20px !important;
}

.gradio-container::before {
    content: '';
    position: fixed;
    inset: 0;
    background-image:
        linear-gradient(rgba(249, 115, 22, 0.03) 1px, transparent 1px),
        linear-gradient(90deg, rgba(249, 115, 22, 0.03) 1px, transparent 1px);
    background-size: 60px 60px;
    pointer-events: none;
    z-index: 0;
}

/* ===== LOGIN VIEW ===== */
#login_view {
    max-width: 660px !important;
    width: 100% !important;
    margin: 20px auto !important;
    background: #12121a !important;
    border: 1px solid #2a2a3d !important;
    border-radius: 20px !important;
    padding: 20px 44px 24px !important;
    box-shadow: 0 20px 60px rgba(0,0,0,0.5) !important;
    position: relative;
    z-index: 1;
}

#login_view > div {
    gap: 4px !important;
}

#login_view .gr-group,
#login_view .gr-block,
#login_view .gr-form,
#login_view .gr-box,
#login_view .gr-panel,
#login_view .gr-row,
#login_view .gr-column,
#login_view .form,
#login_view fieldset {
    margin-bottom: 0 !important;
    padding-bottom: 0 !important;
    background: transparent !important;
    border: none !important;
    outline: none !important;
    box-shadow: none !important;
}

/* ===== CHAT VIEW ===== */
#chat_view {
    position: relative;
    z-index: 1;
    max-width: 1400px !important;
    width: 100% !important;
    margin: 0 auto !important;
    padding: 8px 20px !important;
}

#chat_view .gr-group,
#chat_view .gr-block,
#chat_view .gr-form,
#chat_view .gr-box,
#chat_view .gr-panel,
#chat_view .gr-row,
#chat_view .gr-column,
#chat_view fieldset {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
}

/* ===== TEXTBOX WRAPPERS ===== */
.gradio-container .gr-textbox,
.gradio-container .gr-textbox > div,
.gradio-container .gr-textbox .wrap,
.gradio-container .gr-textbox .wrap-inner,
.gradio-container .gr-textbox .scroll-hide,
.gradio-container .gr-textbox .scroll-hide > div,
.gradio-container .gr-textbox .container,
.gradio-container .gr-textbox .input-container,
.gradio-container .gr-textbox label + div,
.gradio-container .gr-textbox .svelte-1gfkn6j,
.gradio-container .gr-textbox .svelte-1ipelgc,
.gradio-container .gr-textbox .svelte-1lcyrx4,
.gradio-container textarea,
.gradio-container input,
.gradio-container input[type="text"],
.gradio-container input[type="password"],
.gradio-container input[type="email"],
.gradio-container input[type="number"] {
    background: #1a1a28 !important;
    color: #f0f0f5 !important;
    -webkit-text-fill-color: #f0f0f5 !important;
    border-radius: 12px !important;
    box-shadow: none !important;
}

.gradio-container .gr-textbox,
.gradio-container .gr-textbox > div,
.gradio-container .gr-textbox .wrap,
.gradio-container .gr-textbox .wrap-inner,
.gradio-container .gr-textbox .scroll-hide,
.gradio-container .gr-textbox .scroll-hide > div,
.gradio-container .gr-textbox .container,
.gradio-container .gr-textbox .input-container,
.gradio-container .gr-textbox label + div {
    border: none !important;
    outline: none !important;
    padding: 0 !important;
}

.gradio-container .gr-textbox textarea,
.gradio-container .gr-textbox input,
.gradio-container textarea,
.gradio-container input[type="text"],
.gradio-container input[type="password"],
.gradio-container input[type="email"],
.gradio-container input[type="number"] {
    border: 1.5px solid #2a2a3d !important;
    font-size: 14px !important;
    padding: 12px 14px !important;
    transition: all 0.25s ease !important;
    caret-color: #f97316 !important;
    outline: none !important;
    resize: none !important;
    opacity: 1 !important;
}

/* placeholder */
.gradio-container input::placeholder,
.gradio-container textarea::placeholder {
    color: #8f90a6 !important;
    -webkit-text-fill-color: #8f90a6 !important;
    opacity: 1 !important;
}

/* focus */
.gradio-container .gr-textbox:focus-within,
.gradio-container .gr-textbox > div:focus-within,
.gradio-container .gr-textbox .wrap:focus-within,
.gradio-container .gr-textbox .wrap-inner:focus-within,
.gradio-container .gr-textbox .scroll-hide:focus-within,
.gradio-container .gr-textbox .container:focus-within,
.gradio-container .gr-textbox .input-container:focus-within,
.gradio-container .gr-textbox textarea:focus,
.gradio-container .gr-textbox input:focus,
.gradio-container textarea:focus,
.gradio-container input[type="text"]:focus,
.gradio-container input[type="password"]:focus,
.gradio-container input[type="email"]:focus,
.gradio-container input[type="number"]:focus {
    background: #1a1a28 !important;
    color: #f0f0f5 !important;
    -webkit-text-fill-color: #f0f0f5 !important;
    border-color: #f97316 !important;
    box-shadow: 0 0 0 3px rgba(249,115,22,0.10) !important;
    outline: none !important;
}

/* readonly / disabled */
.gradio-container input[readonly],
.gradio-container textarea[readonly],
.gradio-container input:disabled,
.gradio-container textarea:disabled {
    background: #1a1a28 !important;
    color: #f0f0f5 !important;
    -webkit-text-fill-color: #f0f0f5 !important;
    opacity: 1 !important;
    cursor: default !important;
    border: 1.5px solid #2a2a3d !important;
    box-shadow: none !important;
}

/* autofill */
.gradio-container input:-webkit-autofill,
.gradio-container input:-webkit-autofill:hover,
.gradio-container input:-webkit-autofill:focus,
.gradio-container textarea:-webkit-autofill {
    -webkit-text-fill-color: #f0f0f5 !important;
    -webkit-box-shadow: 0 0 0 1000px #1a1a28 inset !important;
    box-shadow: 0 0 0 1000px #1a1a28 inset !important;
    transition: background-color 9999s ease-in-out 0s !important;
    border: 1.5px solid #2a2a3d !important;
    caret-color: #f97316 !important;
}

/* scrollbars */
.gradio-container textarea::-webkit-scrollbar,
.gradio-container .gr-textbox textarea::-webkit-scrollbar {
    width: 8px !important;
    height: 8px !important;
    background: #1a1a28 !important;
}

.gradio-container textarea::-webkit-scrollbar-track,
.gradio-container .gr-textbox textarea::-webkit-scrollbar-track {
    background: #1a1a28 !important;
}

.gradio-container textarea::-webkit-scrollbar-thumb,
.gradio-container .gr-textbox textarea::-webkit-scrollbar-thumb {
    background: #2a2a3d !important;
    border-radius: 10px !important;
    border: 1px solid #1a1a28 !important;
}

.gradio-container textarea,
.gradio-container .gr-textbox textarea {
    scrollbar-color: #2a2a3d #1a1a28 !important;
    scrollbar-width: thin !important;
}

/* labels */
.gradio-container label,
.gradio-container .gr-form label,
.gradio-container .gr-block label {
    color: #8888a0 !important;
    font-size: 13px !important;
    font-weight: 500 !important;
    background: transparent !important;
}

/* ===== BUTTONS ===== */
#login_btn,
#send_btn {
    background: linear-gradient(135deg, #f97316, #ea580c) !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    font-weight: 600 !important;
    font-size: 15px !important;
    padding: 10px 24px !important;
    box-shadow: 0 4px 20px rgba(249,115,22,0.30) !important;
    transition: all 0.25s ease !important;
    cursor: pointer !important;
    min-height: 44px !important;
}

#login_btn:hover,
#send_btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 30px rgba(249,115,22,0.40) !important;
}

#guest_btn {
    background: transparent !important;
    color: #10b981 !important;
    border: 1.5px solid rgba(16,185,129,0.30) !important;
    border-radius: 12px !important;
    font-weight: 600 !important;
    font-size: 15px !important;
    padding: 10px 24px !important;
    transition: all 0.25s ease !important;
    cursor: pointer !important;
    min-height: 44px !important;
}

#guest_btn:hover {
    background: rgba(16,185,129,0.08) !important;
    border-color: #10b981 !important;
    transform: translateY(-2px) !important;
    box-shadow: 0 4px 20px rgba(16,185,129,0.15) !important;
}

#logout_btn {
    background: transparent !important;
    color: #ef4444 !important;
    border: 1.5px solid rgba(239,68,68,0.30) !important;
    border-radius: 12px !important;
    font-weight: 500 !important;
    transition: all 0.25s ease !important;
}

#logout_btn:hover {
    background: rgba(239,68,68,0.08) !important;
    border-color: #ef4444 !important;
}

#clear_btn {
    background: transparent !important;
    color: #8888a0 !important;
    border: 1.5px solid #2a2a3d !important;
    border-radius: 12px !important;
    font-weight: 500 !important;
    transition: all 0.25s ease !important;
}

#clear_btn:hover {
    background: rgba(255,255,255,0.04) !important;
    border-color: #55556a !important;
    color: #f0f0f5 !important;
}

#approve_btn {
    background: linear-gradient(135deg, #10b981, #059669) !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    font-weight: 600 !important;
    box-shadow: 0 4px 20px rgba(16,185,129,0.30) !important;
    transition: all 0.25s ease !important;
}

#approve_btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 30px rgba(16,185,129,0.40) !important;
}

#reject_btn {
    background: transparent !important;
    color: #ef4444 !important;
    border: 1.5px solid rgba(239,68,68,0.30) !important;
    border-radius: 12px !important;
    font-weight: 600 !important;
    transition: all 0.25s ease !important;
}

#reject_btn:hover {
    background: rgba(239,68,68,0.08) !important;
    border-color: #ef4444 !important;
}

/* =========================================================
   CHATBOT — Simple colored bubbles
   Green = user (right)   Blue = agent (left)
   Same approach as the working reference.
   ========================================================= */

/* outer container */
#chatbot {
    background: #12121a !important;
    border: 1px solid #2a2a3d !important;
    border-radius: 16px !important;
    min-height: 350px !important;
    max-height: 55vh !important;
    overflow: hidden !important;
}

/* Gradio fieldset/legend wrappers */
#chatbot fieldset {
    background: #12121a !important;
    border: 1px solid #2a2a3d !important;
    border-radius: 16px !important;
    box-shadow: none !important;
}
#chatbot legend {
    background: #12121a !important;
    color: #8888a0 !important;
    border: 1px solid #2a2a3d !important;
    border-bottom: none !important;
    border-radius: 12px 12px 0 0 !important;
    padding: 6px 12px !important;
    margin-left: 12px !important;
    box-shadow: none !important;
    font-size: 12px !important;
}
#chatbot label,
#chatbot .label-wrap,
#chatbot .heading,
#chatbot .header {
    background: #12121a !important;
    color: #8888a0 !important;
    border-color: #2a2a3d !important;
    box-shadow: none !important;
}

/* --- USER messages — GREEN bubble --- */
.message.user {
    background-color: #22c55e !important;
    border-color: #22c55e !important;
    border-radius: 16px !important;
}
.message.user p,
.message.user span,
.message.user li,
.message.user strong,
.message.user em,
.message.user code,
.message.user a,
.message.user h1, .message.user h2, .message.user h3, .message.user h4,
.message.user td, .message.user th {
    color: #ffffff !important;
}

/* --- BOT messages — BLUE bubble --- */
.message.bot {
    background-color: #2563eb !important;
    border-color: #2563eb !important;
    border-radius: 16px !important;
}
.message.bot p,
.message.bot span,
.message.bot li,
.message.bot strong,
.message.bot em,
.message.bot code,
.message.bot a,
.message.bot h1, .message.bot h2, .message.bot h3, .message.bot h4,
.message.bot td, .message.bot th {
    color: #ffffff !important;
}

/* inline code inside bubbles */
.message.user code,
.message.bot code {
    background: rgba(255,255,255,0.15) !important;
    border-radius: 4px !important;
    padding: 1px 5px !important;
}

/* code blocks inside bubbles */
.message.user pre,
.message.bot pre {
    background: rgba(0,0,0,0.20) !important;
    border-radius: 8px !important;
    padding: 10px 12px !important;
    margin: 6px 0 !important;
}
.message.user pre code,
.message.bot pre code {
    background: transparent !important;
    padding: 0 !important;
}

/* links — slightly lighter for contrast */
.message.user a { color: #dcfce7 !important; text-decoration: underline !important; }
.message.bot a  { color: #bfdbfe !important; text-decoration: underline !important; }

/* footer em tags for metadata */
#chatbot em {
    font-size: 0.85em !important;
    color: #d1d5db !important;
    display: block !important;
    margin-top: 5px !important;
}

/* chatbot scrollbar */
#chatbot ::-webkit-scrollbar { width: 6px; }
#chatbot ::-webkit-scrollbar-track { background: #12121a; }
#chatbot ::-webkit-scrollbar-thumb { background: #2a2a3d; border-radius: 6px; }
#chatbot ::-webkit-scrollbar-thumb:hover { background: #44445a; }

/* ===== SESSION PANEL ===== */
#session_panel {
    background: #12121a !important;
    border: 1px solid #2a2a3d !important;
    border-radius: 16px !important;
    padding: 16px !important;
    min-width: 260px !important;
}

#session_panel .gr-group,
#session_panel .gr-block,
#session_panel .gr-form,
#session_panel .gr-box,
#session_panel .gr-panel,
#session_panel .gr-row,
#session_panel .gr-column,
#session_panel .form,
#session_panel fieldset {
    background: transparent !important;
    border: none !important;
    outline: none !important;
    box-shadow: none !important;
}

#session_panel * {
    color: #f0f0f5 !important;
}

/* ===== ADMIN PANEL ===== */
#admin_panel {
    background: #1a0a0a !important;
    border: 1.5px solid rgba(239,68,68,0.30) !important;
    border-radius: 16px !important;
    padding: 20px !important;
    margin-top: 16px !important;
}

#admin_panel * {
    color: #f0f0f5 !important;
}

/* ===== MARKDOWN ===== */
.gradio-container .markdown-text,
.gradio-container .prose,
.gradio-container .prose p,
.gradio-container .prose li,
.gradio-container .prose span,
.gradio-container .prose strong,
.gradio-container .prose h1,
.gradio-container .prose h2,
.gradio-container .prose h3,
.gradio-container .prose h4 {
    color: #f0f0f5 !important;
    background: transparent !important;
}

/* ===== COMPACT BRAND ===== */
.brand-compact {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 14px;
    padding: 8px 0 4px;
}

.brand-compact .shoe-icon {
    font-size: 36px;
}

.brand-compact .brand-text h1 {
    font-size: 26px;
    font-weight: 800;
    color: #f0f0f5 !important;
    margin: 0;
    line-height: 1.2;
}

.brand-compact .brand-text .tagline {
    font-size: 11px;
    color: #8888a0 !important;
    letter-spacing: 2px;
    text-transform: uppercase;
    margin: 0;
    background: transparent !important;
}

.or-divider {
    display: flex;
    align-items: center;
    gap: 16px;
    margin: 8px 0;
    color: #55556a;
    font-size: 11px;
    font-weight: 500;
    letter-spacing: 1px;
    text-transform: uppercase;
}

.or-divider::before,
.or-divider::after {
    content: '';
    flex: 1;
    height: 1px;
    background: linear-gradient(90deg, transparent, #2a2a3d, transparent);
}

.features-row {
    display: flex;
    justify-content: center;
    gap: 40px;
    padding: 10px 0 4px;
    background: transparent !important;
}

.features-row .feat {
    font-size: 12px;
    color: #55556a !important;
}

.status-badge {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 5px 12px;
    background: rgba(16,185,129,0.10);
    border: 1px solid rgba(16,185,129,0.20);
    border-radius: 100px;
    font-size: 11px;
    color: #10b981 !important;
    font-weight: 500;
}

.status-badge .dot {
    width: 6px;
    height: 6px;
    background: #10b981;
    border-radius: 50%;
    display: inline-block;
    animation: pulse 2s ease-in-out infinite;
}

@keyframes pulse {
    0%, 100% { opacity: 1; }
    50% { opacity: 0.4; }
}

.footer-text {
    text-align: center;
    font-size: 11px;
    color: #55556a !important;
    padding: 4px 0;
    background: transparent !important;
}

.footer-text span {
    color: #f97316 !important;
    font-weight: 600;
}

footer {
    display: none !important;
}
"""

In [ ]:

BRAND_HTML = """
<div class="brand-compact">
    <span class="shoe-icon">👟</span>
    <div class="brand-text">
        <h1>ShoeOps Agent</h1>
        <p class="tagline">Inventory &bull; Orders &bull; Intelligence</p>
    </div>
</div>
"""

FEATURES_HTML = """
<div class="features-row">
    <span class="feat">📦 Real-time Inventory</span>
    <span class="feat">🔍 Smart Search</span>
    <span class="feat">🛡️ Secure Transfers</span>
</div>
"""

FOOTER_HTML = '<div class="footer-text">Powered by <span>LangGraph</span> &bull; AI-Assisted Operations</div>'

CHAT_HEADER_HTML = """
<div style="display:flex; align-items:center; justify-content:space-between; padding:4px 0 12px;">
    <div style="display:flex; align-items:center; gap:10px;">
        <span style="font-size:28px;">👟</span>
        <span style="font-size:20px; font-weight:700; color:#f0f0f5;">ShoeOps Agent</span>
    </div>
    <div class="status-badge"><span class="dot"></span> Agent Online</div>
</div>
"""

In [ ]:

with gr.Blocks(title="ZAMSH ShoeOps Agent", css=CUSTOM_CSS, theme=gr.themes.Base()) as demo:

    session_state = gr.State({})
    thread_id = gr.State(str(uuid4()))
    history_state = gr.State([])

    login_view = gr.Column(visible=True, elem_id="login_view")
    chat_view = gr.Column(visible=False, elem_id="chat_view")

    # =================== LOGIN SCREEN ===================
    with login_view:
        gr.HTML(BRAND_HTML)

        with gr.Row():
            u = gr.Textbox(label="Username", placeholder="Enter your username", scale=1)
            p = gr.Textbox(label="PIN Code", type="password", placeholder="••••", scale=1)

        with gr.Row():
            login_btn = gr.Button("🔐  Login", elem_id="login_btn", scale=1)
            guest_btn = gr.Button("Browse as Guest  →", elem_id="guest_btn", scale=1)

        login_status = gr.Markdown("")

        gr.HTML(FEATURES_HTML)
        gr.HTML(FOOTER_HTML)

    # =================== CHAT SCREEN ===================
    with chat_view:
        gr.HTML(CHAT_HEADER_HTML)

        with gr.Row():
            with gr.Column(scale=4):
                chatbot = gr.Chatbot(label="Chat", elem_id="chatbot", height=350)

                with gr.Row():
                    user_in = gr.Textbox(placeholder="Type your message...", show_label=False, scale=8, container=False)
                    send_btn = gr.Button("Send ➤", elem_id="send_btn", scale=1, min_width=120)

                with gr.Row():
                    clear_btn = gr.Button("🔄 New Chat", elem_id="clear_btn", scale=1)
                    logout_btn = gr.Button("← Logout", elem_id="logout_btn", scale=1)

            with gr.Column(scale=1, min_width=280, elem_id="session_panel"):
                gr.Markdown('<div class="session-title">📋 Session Info</div>')
                display_name_out = gr.Textbox(label="Display Name", interactive=False, value="Guest")
                role_out = gr.Textbox(label="Role", interactive=False, value="customer")
                customer_id_out = gr.Textbox(label="Customer ID", interactive=False, value="")
                tool_log = gr.Markdown("🔧 Tools used: None")

        admin_panel = gr.Group(visible=False, elem_id="admin_panel")
        with admin_panel:
            gr.Markdown("## 🛑 Manager Review Required")
            draft_json = gr.Textbox(label="Draft action (JSON)", lines=10)
            admin_feedback = gr.Textbox(label="Manager feedback (optional)", lines=3)
            with gr.Row():
                approve_btn = gr.Button("✅ Approve", elem_id="approve_btn")
                reject_btn = gr.Button("❌ Reject", elem_id="reject_btn")

    # =================== WIRING ===================

    login_btn.click(
        login_handler,
        inputs=[u, p, session_state, thread_id, history_state],
        outputs=[login_status, session_state, thread_id, history_state, login_view, chat_view, display_name_out, role_out, customer_id_out],
    ).then(lambda h: h, inputs=[history_state], outputs=[chatbot])

    u.submit(
        login_handler,
        inputs=[u, p, session_state, thread_id, history_state],
        outputs=[login_status, session_state, thread_id, history_state, login_view, chat_view, display_name_out, role_out, customer_id_out],
    ).then(lambda h: h, inputs=[history_state], outputs=[chatbot])

    p.submit(
        login_handler,
        inputs=[u, p, session_state, thread_id, history_state],
        outputs=[login_status, session_state, thread_id, history_state, login_view, chat_view, display_name_out, role_out, customer_id_out],
    ).then(lambda h: h, inputs=[history_state], outputs=[chatbot])

    guest_btn.click(
        guest_handler,
        inputs=[session_state, thread_id],
        outputs=[session_state, thread_id, history_state, login_view, chat_view, display_name_out, role_out, customer_id_out],
    ).then(lambda h: h, inputs=[history_state], outputs=[chatbot])

    send_btn.click(
        chat_handler,
        inputs=[user_in, history_state, thread_id, session_state],
        outputs=[user_in, history_state, thread_id, admin_panel, draft_json, admin_feedback, tool_log, session_state, display_name_out, role_out, customer_id_out, login_view, chat_view],
    ).then(lambda h: h, inputs=[history_state], outputs=[chatbot])

    user_in.submit(
        chat_handler,
        inputs=[user_in, history_state, thread_id, session_state],
        outputs=[user_in, history_state, thread_id, admin_panel, draft_json, admin_feedback, tool_log, session_state, display_name_out, role_out, customer_id_out, login_view, chat_view],
    ).then(lambda h: h, inputs=[history_state], outputs=[chatbot])

    clear_btn.click(
        new_chat,
        inputs=[],
        outputs=[thread_id, history_state, admin_panel, draft_json, admin_feedback, tool_log],
    ).then(lambda h: h, inputs=[history_state], outputs=[chatbot])

    approve_btn.click(
        lambda h, t: admin_decide_handler("APPROVE", "", h, t),
        inputs=[history_state, thread_id],
        outputs=[history_state, thread_id, admin_panel, draft_json, admin_feedback, tool_log],
    ).then(lambda h: h, inputs=[history_state], outputs=[chatbot])

    reject_btn.click(
        lambda fb, h, t: admin_decide_handler("REJECT", fb, h, t),
        inputs=[admin_feedback, history_state, thread_id],
        outputs=[history_state, thread_id, admin_panel, draft_json, admin_feedback, tool_log],
    ).then(lambda h: h, inputs=[history_state], outputs=[chatbot])

    logout_btn.click(
        logout_handler,
        inputs=[session_state],
        outputs=[login_status, session_state, thread_id, history_state, login_view, chat_view, display_name_out, role_out, customer_id_out, u, p],
    ).then(lambda h: h, inputs=[history_state], outputs=[chatbot])

/tmp/ipykernel_491/3882429997.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="ZAMSH ShoeOps Agent", css=CUSTOM_CSS, theme=gr.themes.Base()) as demo:


In [ ]:
# Block 8 — Launch
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b3798bcd09290ed239.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
